# Enterprise Data Agent Workshop

**Build a memory-aware enterprise data agent on Oracle AI Database 26ai**

This notebook walks through the construction of an **agent harness** — the loop on top of an LLM that gives it persistent memory, tool dispatch, retrieval, and identity-aware authorization. Every primitive lives inside Oracle AI Database: in-DB ONNX embeddings, hybrid vector + Oracle Text retrieval, a JavaScript MLE sandbox, and Deep Data Security row policies.

The whole loop is roughly 300 lines of Python; the rest is database primitives.

> 📖 **Workshop guide:** [`README.md`](../README.md). Each Part below has its own deep-dive in [`docs/`](../docs/).

![Enterprise Data Agent — Oracle-Native Architecture](../images/cover-oracle-native-arch.png)

| Part | Topic | Guide |
|---|---|---|
| 1 | Setup & connectivity (pre-built) | [Part 1](../docs/part-1-setup.md) |
| 2 | Long-term memory with OAMP | [Part 2](../docs/part-2-oamp-memory.md) |
| 3 | Retrieval strategies (vector + hybrid RRF) | [Part 3](../docs/part-3-retrieval.md) |
| 4 | Tools & skills (vector-indexed registries) | [Part 4](../docs/part-6-tools-and-skills.md) |
| 5 | The agent loop | [Part 5](../docs/part-7-agent-loop.md) |
| 6 | Identity-aware authorization with DDS | [Part 6](../docs/part-8-dds-identity.md) |

> 📋 **[TODO checklist](../docs/TODO-checklist.md)** — 9 hands-on tasks at a glance.

> ℹ️ **This is the full-source `complete_with_setup_code` notebook.** It mirrors the original 9-part source `enterprise_data_agent.ipynb` and **includes every Oracle DDL** (AGENT user creation, vector_memory_size, ONNX downloads, SUPPLYCHAIN seed, DDS policies, scheduler job, etc.).
>
> Use this notebook when:
> - You want to deploy the harness against an Oracle that **isn't** the workshop Codespace (the workshop's pre-built setup scripts won't run for you).
> - You want to see *how* every primitive is set up, end-to-end.
>
> If you just want to **learn the harness pattern**, use [`notebook_student.ipynb`](notebook_student.ipynb) (which has 11 parts following the new structure — Setup, OAMP, Retrieval, **DBFS**, **MLE**, Tools & Skills, Agent Loop, DDS, Duality Views, Scheduler, Tool-Output Offload) and refer to the [Part guides in `docs/`](../docs/). This file's internal part numbering predates that restructure.



## At a glance — the 9 TODOs and the running app

**9 hands-on coding TODOs across 11 parts.** This is the *full-source* notebook — it includes every Oracle DDL, every bootstrap step, and every TODO solution inline. The student notebook strips the setup so you focus on the 9 harness TODOs; this notebook is for when you want to deploy the harness against an Oracle that *isn't* the workshop Codespace.

The 9 TODOs map to the student notebook's numbering. Setup steps (open `agent_conn`, register OAMP identities) and demo cells (three-way retrieval probe, two-identity comparison) appear as **Setup checkpoint** / **Demo** in this notebook rather than as numbered TODOs.

| Part | Topic | TODO(s) |
|---|---|---|
| 1 | Setup & connectivity | — |
| 2 | Long-term memory with OAMP | **TODO 1** — `_scan_tables` |
| 3 | Retrieval (vector + hybrid RRF) | **TODO 2** — `retrieve_knowledge`<br>**TODO 3** — `hybrid_rrf_search_memories` |
| 4 | DBFS scratchpad | — |
| 5 | Oracle MLE compute sandbox | — |
| 6 | Tools & skills | **TODO 4** — register `tool_run_sql` |
| 7 | The agent loop | **TODO 5** — `agent_turn` |
| 8 | Identity-aware authorization (DDS) | **TODO 6** — `set_identity` |
| 9 | JSON Relational Duality Views | **TODO 7** — `tool_get_document` |
| 10 | Continuous scans via `DBMS_SCHEDULER` | — |
| 11 | Tool-output offload | **TODO 8** — `log_tool`<br>**TODO 9** — `tool_fetch_tool_output` |

> Full checklist: [`docs/TODO-checklist.md`](../docs/TODO-checklist.md).

### From notebook concept → running application

**The same harness lives in a Flask + React app** against the *same* Oracle, *same* OAMP store, *same* `toolbox` and `skillbox`. The notebook teaches the concept; the app shows it deployed.

> **Open the running app now:** [http://localhost:3000](http://localhost:3000) — the Codespace auto-forwards port 3000 and opens a preview when the UI is ready. (If it didn't open, check the **PORTS** tab in VS Code.)
>
> Architecture: [`app/README.md`](../app/README.md). Bootstrap source: `app/scripts/bootstrap.py` / `seed.py` / `setup_advanced.py`.

# Part 1 — Setup & Connectivity

> 📖 **Guide:** [`docs/part-1-setup.md`](../docs/part-1-setup.md)

The setup cells below are **pre-built**. They boot Oracle AI Database (if needed), create the `AGENT` user, configure `vector_memory_size` and `pga_aggregate_limit`, load the ONNX embedder and reranker into the database, and wire up the chat LLM client.

Run them once. They are idempotent — re-running is safe.

> ⚠️ **First run only:** if you see `ACTION REQUIRED: bounce the database` in the output, run `docker restart oracle-free` in a terminal, wait ~60s, and restart this kernel. Subsequent runs detect that `vector_memory_size` is configured and skip the bounce step.

In [ ]:
# 1.0 — Imports. Dependencies are pre-installed in the Codespace; uncomment to install locally.
# %pip install --quiet "oracledb>=2.4" "openai>=1.50" "numpy>=1.26" "tqdm>=4.66" "oracleagentmemory>=26.4" "gdown>=5.2"

import os, time, json, getpass, warnings, subprocess, urllib.request, urllib.error
import tempfile, zipfile, pathlib, shutil, re, hashlib, uuid
import numpy as np
import oracledb
from dataclasses import dataclass

warnings.filterwarnings("ignore", category=UserWarning, module="oracleagentmemory")
print("imports OK — oracledb", oracledb.__version__)

In [ ]:
# 1.1 — Credentials. LLM_PROVIDER picks the chat backend; OPENAI_API_KEY is also
# used by the OAMP extraction LLM regardless of which chat provider you pick.

LLM_PROVIDER = os.environ.setdefault("LLM_PROVIDER", "oci").lower()  # 'oci' or 'openai'
assert LLM_PROVIDER in ("openai", "oci"), "LLM_PROVIDER must be 'openai' or 'oci'"

if LLM_PROVIDER == "oci":
    if not os.environ.get("OCI_GENAI_API_KEY"):
        os.environ["OCI_GENAI_API_KEY"] = getpass.getpass("OCI GenAI API key: ")
    os.environ.setdefault("OCI_GENAI_ENDPOINT",
                          "https://inference.generativeai.us-phoenix-1.oci.oraclecloud.com")
    os.environ.setdefault("LLM_MODEL", "xai.grok-4-1-fast-reasoning")
    print(f"Provider: OCI GenAI ({os.environ['LLM_MODEL']})")
else:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    os.environ.setdefault("LLM_MODEL", "gpt-5.5")
    print(f"Provider: OpenAI ({os.environ['LLM_MODEL']})")

In [ ]:
# 1.2 — Connection helper. Thin retry wrapper around oracledb.connect — used everywhere.

SYS_DSN    = "localhost:1521/FREEPDB1"
SYS_USER   = "sys"
SYS_PASS   = "OraclePwd_2025"
AGENT_USER = "AGENT"
AGENT_PASS = "AgentPwd_2025"


def connect(user: str, password: str, dsn: str, mode: int | None = None,
            retries: int = 5) -> oracledb.Connection:
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            kwargs = dict(user=user, password=password, dsn=dsn)
            if mode is not None:
                kwargs["mode"] = mode
            conn = oracledb.connect(**kwargs)
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE rownum = 1")
                print(f"connected as {user}@{dsn} — {cur.fetchone()[0]}")
            return conn
        except Exception as e:
            last_err = e
            print(f"  attempt {attempt}/{retries} failed: {e}")
            time.sleep(3)
    raise RuntimeError(f"could not connect to {dsn}: {last_err}")

In [ ]:
# 1.3 — Bootstrap (SYS): create AGENT user, allocate vector memory, raise PGA.
# Idempotent — safe to re-run. May print 'ACTION REQUIRED: bounce' on first run.

sys_conn = connect(SYS_USER, SYS_PASS, SYS_DSN, mode=oracledb.AUTH_MODE_SYSDBA)

bootstrap_stmts = [
    f"DECLARE n NUMBER; BEGIN "
    f"  SELECT COUNT(*) INTO n FROM all_users WHERE username = '{AGENT_USER}'; "
    f"  IF n = 0 THEN EXECUTE IMMEDIATE 'CREATE USER {AGENT_USER} IDENTIFIED BY {AGENT_PASS}'; END IF; "
    f"END;",
    f"GRANT CONNECT, RESOURCE, CREATE SESSION TO {AGENT_USER}",
    f"GRANT CREATE TABLE, CREATE SEQUENCE, CREATE VIEW, CREATE PROCEDURE TO {AGENT_USER}",
    f"GRANT UNLIMITED TABLESPACE TO {AGENT_USER}",
    f"GRANT SELECT ON SYS.V_$SQL TO {AGENT_USER}",
    f"GRANT SELECT_CATALOG_ROLE TO {AGENT_USER}",
    f"GRANT EXECUTE ON DBMS_MLE TO {AGENT_USER}",
    f"GRANT DB_DEVELOPER_ROLE TO {AGENT_USER}",
    f"GRANT EXECUTE DYNAMIC MLE TO {AGENT_USER}",
    f"GRANT CTXAPP TO {AGENT_USER}",
    f"GRANT CREATE MINING MODEL TO {AGENT_USER}",
    f"GRANT SELECT ANY TABLE TO {AGENT_USER}",
]

with sys_conn.cursor() as cur:
    for stmt in bootstrap_stmts:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            if e.args[0].code not in (1031,):
                print("  skip:", str(e).splitlines()[0][:80])
sys_conn.commit()
print("AGENT user provisioned with required grants.")

# Vector memory pool — required for HNSW vector indexes.
TARGET_VECTOR_MEMORY = 512 * 1024 * 1024
with sys_conn.cursor() as cur:
    cur.execute("SELECT value FROM v$parameter WHERE name = 'vector_memory_size'")
    current = int(cur.fetchone()[0] or 0)
if current >= TARGET_VECTOR_MEMORY:
    print(f"vector_memory_size already configured ({current/1024/1024:.0f} MiB) — OK.")
else:
    with sys_conn.cursor() as cur:
        cur.execute("ALTER SYSTEM SET vector_memory_size = 512M SCOPE=SPFILE CONTAINER=ALL")
    sys_conn.commit()
    print("vector_memory_size set to 512M (SPFILE).")
    print("ACTION REQUIRED: docker restart oracle-free, then restart this kernel.")

# PGA limit — DBMS_VECTOR.RERANK needs more than the Free image default.
TARGET_PGA = 4 * 1024 * 1024 * 1024
with sys_conn.cursor() as cur:
    cur.execute("SELECT value FROM v$parameter WHERE name = 'pga_aggregate_limit'")
    pga_now = int(cur.fetchone()[0] or 0)
if pga_now < TARGET_PGA:
    try:
        with sys_conn.cursor() as cur:
            cur.execute("ALTER SYSTEM SET pga_aggregate_limit = 4G SCOPE=BOTH")
        sys_conn.commit()
        print("pga_aggregate_limit raised to 4 GiB.")
    except oracledb.DatabaseError as e:
        print(f"  pga_aggregate_limit raise failed (ORA-{e.args[0].code:05d}); "
              "reranker may fall back to cosine ordering under load.")
else:
    print(f"pga_aggregate_limit already configured ({pga_now/1024/1024/1024:.1f} GiB).")

## Connect as the `AGENT` user

> 📖 **See:** [Part 1 guide → Setup checkpoint 1](../docs/part-1-setup.md#todo-1-connect-to-oracle-as-the-agent-user)

The pre-built `connect()` function takes `(user, password, dsn)`. Use it to open `agent_conn` — the connection every later cell uses.

The constants `AGENT_USER`, `AGENT_PASS`, and `SYS_DSN` are already defined.


In [ ]:
# Setup checkpoint 1 open `agent_conn` as the AGENT user.
# Hint: agent_conn = connect(AGENT_USER, AGENT_PASS, SYS_DSN)

agent_conn = connect(AGENT_USER, AGENT_PASS, SYS_DSN)

In [ ]:
# ✅ Checkpoint: Setup checkpoint 1assert "agent_conn" in dir() and agent_conn is not None, (
    "❌ Setup checkpoint 1incomplete — agent_conn is not defined. See docs/part-1-setup.md."
)
with agent_conn.cursor() as cur:
    cur.execute("SELECT user FROM dual")
    assert cur.fetchone()[0] == "AGENT", "Connected as wrong user — should be AGENT."
print("✅ Setup checkpoint 1passed — agent_conn is the AGENT user")

### Pre-built — load ONNX embedder and reranker into Oracle

The cell below downloads two ONNX models, copies them into the `oracle-free` container, and registers them in Oracle's mining-model registry. After it runs:

- `VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :text AS DATA)` — 384-dim sentence embedder, called from SQL.
- `PREDICTION(RERANKER_ONNX USING :q AS DATA1, :doc AS DATA2)` — cross-encoder reranker, called from SQL.

Both models live inside the database — same trust boundary as your data, no network round-trips for embedding.

In [ ]:
# 1.4 — Download + load ONNX embedder. Idempotent.

CONTAINER_NAME       = "oracle-free"
CONTAINER_MODEL_DIR  = "/opt/oracle/onnx_models"
ONNX_FILE            = "all_MiniLM_L12_v2.onnx"
ONNX_DIRECTORY       = "ONNX_DIR"
ONNX_EMBED_MODEL     = "ALL_MINILM_L12_V2"
ORACLE_MODEL_URL = (
    "https://adwc4pm.objectstorage.us-ashburn-1.oci.customer-oci.com"
    "/p/TtH6hL2y25EypZ0-rrczRZ1aXp7v1ONbRBfCiT-BDBN8WLKQ3lgyW6RxCfIFLdA6"
    "/n/adwc4pm/b/OML-ai-models/o/all_MiniLM_L12_v2_augmented.zip"
)
os.environ["PATH"] = ":".join(["/opt/homebrew/bin", "/usr/local/bin", "/usr/bin", "/bin",
                                os.environ.get("PATH", "")])
CONTAINER_CLI = "docker" if shutil.which("docker") else "podman"


def _exists_in_container(path):
    return subprocess.run(
        [CONTAINER_CLI, "exec", CONTAINER_NAME, "test", "-f", path],
        capture_output=True,
    ).returncode == 0


subprocess.run([CONTAINER_CLI, "exec", CONTAINER_NAME, "mkdir", "-p", CONTAINER_MODEL_DIR],
               check=True)

target_path = f"{CONTAINER_MODEL_DIR}/{ONNX_FILE}"
if _exists_in_container(target_path):
    print(f"'{ONNX_FILE}' already in {CONTAINER_NAME} — skipping download.")
else:
    print("Downloading Oracle augmented all-MiniLM-L12-v2 ONNX model (~117 MB)...")
    with tempfile.TemporaryDirectory() as tmp:
        zip_path = os.path.join(tmp, "model.zip")
        urllib.request.urlretrieve(ORACLE_MODEL_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmp)
        onnx_path = next(pathlib.Path(tmp).glob("*.onnx"))
        subprocess.run([CONTAINER_CLI, "cp", str(onnx_path),
                        f"{CONTAINER_NAME}:{target_path}"], check=True)
        subprocess.run([CONTAINER_CLI, "exec", "--user", "0", CONTAINER_NAME,
                        "chmod", "644", target_path], check=False, capture_output=True)
    print(f"  copied to {CONTAINER_NAME}:{target_path}")

# Map an Oracle directory at the container path so DBMS_VECTOR.LOAD_ONNX_MODEL can read it.
with sys_conn.cursor() as cur:
    cur.execute(f"CREATE OR REPLACE DIRECTORY {ONNX_DIRECTORY} AS '{CONTAINER_MODEL_DIR}'")
    cur.execute(f"GRANT READ, WRITE ON DIRECTORY {ONNX_DIRECTORY} TO {AGENT_USER}")
sys_conn.commit()

# Register the embedder.
with agent_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
                m=ONNX_EMBED_MODEL)
    already_loaded = cur.fetchone()[0] > 0

if already_loaded:
    print(f"model {ONNX_EMBED_MODEL!r} already loaded.")
else:
    print(f"loading ONNX model {ONNX_EMBED_MODEL!r}...")
    with agent_conn.cursor() as cur:
        cur.execute(
            "BEGIN DBMS_VECTOR.LOAD_ONNX_MODEL("
            "  directory => :d, file_name => :f, model_name => :m, "
            "  metadata => JSON('{\"function\":\"embedding\",\"embeddingOutput\":\"embedding\","
            "  \"input\":{\"input\":[\"DATA\"]}}')); END;",
            d=ONNX_DIRECTORY, f=ONNX_FILE, m=ONNX_EMBED_MODEL,
        )
    agent_conn.commit()
    print(f"loaded {ONNX_EMBED_MODEL!r}.")

# Smoke test — produce one embedding and report dimension.
with agent_conn.cursor() as cur:
    cur.execute(f"SELECT VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :t AS DATA) FROM dual",
                t="Oracle in-database embedding round-trip.")
    vec = cur.fetchone()[0]
ONNX_EMBED_DIM = len(vec)
print(f"VECTOR_EMBEDDING produced a {ONNX_EMBED_DIM}-dim vector.")

In [ ]:
# 1.5 — Download + load ONNX reranker (optional but recommended).
# If RERANKER_URL is empty, the rerank() helper falls back to cosine ordering.

DEFAULT_RERANKER_URL = (
    "https://drive.google.com/file/d/1-xDRSHr_ulbO7MqVlWLu6ZA2J-bCjUoY/view?usp=drive_link"
)
RERANKER_MODEL = os.environ.get("RERANKER_MODEL_NAME", "RERANKER_ONNX")
RERANKER_URL   = os.environ.get("RERANKER_URL", DEFAULT_RERANKER_URL).strip()
RERANKER_FILE  = os.environ.get("RERANKER_FILE", "bge_reranker_base.onnx")


def _reranker_loaded() -> bool:
    with agent_conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM user_mining_models WHERE model_name = :m",
                    m=RERANKER_MODEL)
        return cur.fetchone()[0] > 0


if _reranker_loaded():
    print(f"reranker {RERANKER_MODEL!r} already loaded.")
elif not RERANKER_URL:
    print("RERANKER_URL not set — skipping reranker load. rerank() will pass through.")
else:
    target = f"{CONTAINER_MODEL_DIR}/{RERANKER_FILE}"
    if not _exists_in_container(target):
        print(f"downloading reranker from {RERANKER_URL}...")
        with tempfile.TemporaryDirectory() as tmp:
            local_path = os.path.join(tmp, RERANKER_FILE)
            drive_match = re.search(r"drive\.google\.com/.*?/d/([A-Za-z0-9_-]+)", RERANKER_URL)
            if drive_match:
                import gdown
                gdown.download(id=drive_match.group(1), output=local_path, quiet=False)
            else:
                urllib.request.urlretrieve(RERANKER_URL, local_path)
            src_path = local_path
            if local_path.endswith(".zip"):
                with zipfile.ZipFile(local_path) as zf:
                    zf.extractall(tmp)
                src_path = str(next(pathlib.Path(tmp).glob("*.onnx")))
            subprocess.run([CONTAINER_CLI, "cp", src_path, f"{CONTAINER_NAME}:{target}"], check=True)
            subprocess.run([CONTAINER_CLI, "exec", "--user", "0", CONTAINER_NAME,
                            "chmod", "644", target], check=False, capture_output=True)
    rerank_meta = json.dumps({
        "function": "regression", "regressionOutput": "output",
        "input": {"first_input": ["DATA1"], "second_input": ["DATA2"]},
    })
    print(f"loading reranker {RERANKER_MODEL!r}...")
    with agent_conn.cursor() as cur:
        cur.execute(
            "BEGIN DBMS_VECTOR.LOAD_ONNX_MODEL("
            "  directory => :d, file_name => :f, model_name => :m, metadata => JSON(:meta)); END;",
            d=ONNX_DIRECTORY, f=RERANKER_FILE, m=RERANKER_MODEL, meta=rerank_meta,
        )
    agent_conn.commit()
    print(f"loaded {RERANKER_MODEL!r}.")

print(f"reranker available: {_reranker_loaded()}")

In [ ]:
# 1.6 — rerank() helper. Calls Oracle's PREDICTION() against the loaded reranker;
# falls through to cosine ordering when no model is loaded so call sites are uniform.

def rerank(query: str, candidates: list[dict], top_k: int = 5,
           content_key: str = "body") -> list[dict]:
    if not candidates:
        return candidates
    if not _reranker_loaded():
        return candidates[:top_k]

    docs = [{"index": i, "content": str(c.get(content_key, ""))[:1500]}
            for i, c in enumerate(candidates)]

    sql = (
        f"SELECT t.idx, "
        f"       PREDICTION({RERANKER_MODEL} USING :q AS DATA1, t.content AS DATA2) AS score "
        "  FROM JSON_TABLE(:docs, '$[*]' COLUMNS ("
        "         idx     NUMBER          PATH '$.index', "
        "         content VARCHAR2(4000)  PATH '$.content'"
        "       )) t "
        " ORDER BY score DESC FETCH FIRST :k ROWS ONLY"
    )
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, q=query, docs=json.dumps(docs), k=top_k)
            ranked = list(cur)
    except oracledb.DatabaseError as e:
        print(f"  rerank failed — falling back to cosine order: {e}")
        return candidates[:top_k]

    out = []
    for idx, score in ranked:
        if idx is None or int(idx) >= len(candidates):
            continue
        item = dict(candidates[int(idx)])
        item["rerank_score"] = float(score) if score is not None else 0.0
        out.append(item)
    return out


# Smoke test
_demo = [
    {"body": "Oracle AI Database supports vector search natively."},
    {"body": "Bananas are yellow."},
    {"body": "PREDICTION() rescores documents against a query using a loaded reranker."},
]
print("rerank smoke test:")
for hit in rerank("how does in-database reranking work?", _demo, top_k=3):
    print(f"  {hit.get('rerank_score', 0.0):>10.4f}  {hit['body']}")

In [ ]:
# 1.7 — Chat LLM client. Uses the standard openai SDK pointed at OpenAI directly,
# or at OCI GenAI's OpenAI-compatible endpoint depending on LLM_PROVIDER.

from openai import OpenAI

LLM_MODEL = os.environ["LLM_MODEL"]

if LLM_PROVIDER == "oci":
    llm = OpenAI(
        base_url=os.environ["OCI_GENAI_ENDPOINT"],
        api_key=os.environ["OCI_GENAI_API_KEY"],
    )
else:
    llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


def chat(messages: list[dict], tools: list[dict] | None = None,
         model: str = LLM_MODEL, max_retries: int = 3):
    """Call the LLM with retry on 429."""
    kwargs = {"model": model, "messages": messages}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    delay = 2.0
    for attempt in range(max_retries + 1):
        try:
            return llm.chat.completions.create(**kwargs)
        except Exception as e:
            status = getattr(e, "status_code", None) or 0
            if status == 429 and attempt < max_retries:
                print(f"  429 — retrying in {delay:.0f}s")
                time.sleep(delay); delay *= 2
            else:
                raise


resp = chat([
    {"role": "system", "content": "Be terse."},
    {"role": "user",   "content": "Say 'pong'."},
])
print(f"[{LLM_PROVIDER}/{LLM_MODEL}]", resp.choices[0].message.content)

# Part 2 — Long-Term Memory with OAMP

> 📖 **Guide:** [`docs/part-2-oamp-memory.md`](../docs/part-2-oamp-memory.md)

> 🔧 **TODO in this part:** **TODO 1** — `_scan_tables`

The long-term store **is** the [Oracle AI Agent Memory Package (OAMP)](https://www.oracle.com/database/ai-agent-memory/). Instead of hand-rolling memory tables, we hand a connection to `OracleAgentMemory` and let it own the DDL, the embedding pipeline, and the retrieval surface.

| OAMP primitive | What it stores |
|---|---|
| `memory` | Durable facts — scanned schema entries, corrections, tool outputs |
| `thread` | A conversation. Holds messages, exposes a context card |
| `context_card` | Compact, query-relevant block of memories + recent turns |

We keep one bespoke table — `scan_history` — for **procedural** memory of the agent's own scans (queried by time/owner, not by meaning).

In [ ]:
# 2.1 — In-DB ONNX embedder for OAMP. Every embed() call issues a SELECT VECTOR_EMBEDDING
# against agent_conn — zero network calls, embedding happens in the same process as the database work.

from oracleagentmemory.core import OracleAgentMemory
from oracleagentmemory.core.llms import Llm
from oracleagentmemory.apis.embedders.embedder import IEmbedder


class OracleONNXEmbedder(IEmbedder):
    def __init__(self, conn, model_name: str = ONNX_EMBED_MODEL, dim: int = ONNX_EMBED_DIM):
        self._conn = conn
        self._model = model_name
        self._dim = dim

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        out = np.zeros((len(texts), self._dim), dtype=np.float32)
        sql = f"SELECT VECTOR_EMBEDDING({self._model} USING :t AS DATA) FROM dual"
        with self._conn.cursor() as cur:
            for i, t in enumerate(texts):
                cur.execute(sql, t=t)
                out[i] = np.asarray(list(cur.fetchone()[0]), dtype=np.float32)
        return out

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return self.embed(texts, is_query=is_query)


# Wire OAMP's extraction LLM to whichever chat provider §1 picked.
if LLM_PROVIDER == "oci":
    extraction_llm = Llm(f"openai/{LLM_MODEL}",
                         api_base=os.environ["OCI_GENAI_ENDPOINT"],
                         api_key=os.environ["OCI_GENAI_API_KEY"])
else:
    extraction_llm = Llm(LLM_MODEL)

memory_client = OracleAgentMemory(
    connection=agent_conn,
    embedder=OracleONNXEmbedder(agent_conn),
    llm=extraction_llm,
    extract_memories=True,
    schema_policy="create_if_necessary",
    table_name_prefix="eda_onnx_",
)
print("OAMP client wired (in-DB ONNX embedder + extraction LLM).")

## Register the operator and agent identities

> 📖 **See:** [Part 2 guide → Setup checkpoint 2](../docs/part-2-oamp-memory.md#todo-2-register-the-user-and-agent-identities)

Every memory record OAMP stores carries a `user_id` and an `agent_id`. Register both up front so the rest of the notebook can reference them by stable strings.

Wrap each call in `try/except ValueError` so re-running doesn't blow up on "already exists".


In [ ]:
USER_ID  = "enterprise-operator"
AGENT_ID = "enterprise-data-agent"

# Setup checkpoint 2 Register USER_ID and AGENT_ID with OAMP.
# Use memory_client.add_user(USER_ID, info) and memory_client.add_agent(AGENT_ID, info).
# Wrap each in try/except ValueError so re-runs don't fail on "already exists".

for register_fn, eid, info in [
    (memory_client.add_user,  USER_ID,  "Operator querying the enterprise database in natural language."),
    (memory_client.add_agent, AGENT_ID, "Data agent grounded in scanned schema metadata."),
]:
    try:
        register_fn(eid, info)
        print(f"registered {eid}")
    except ValueError as e:
        if "already exists" in str(e):
            print(f"(already exists) {eid}")
        else:
            raise

In [ ]:
# ✅ Checkpoint: Setup checkpoint 2_users = memory_client._store.list_users() if hasattr(memory_client._store, "list_users") else None
print("✅ Setup checkpoint 2passed — user/agent registered with OAMP")

In [ ]:
# 2.2 — scan_history: procedural memory of the agent's own scans.
# Queried by time/owner, not by meaning, so it stays a regular table (not OAMP).

with agent_conn.cursor() as cur:
    try:
        cur.execute(
            "CREATE TABLE scan_history ("
            "  scan_id          VARCHAR2(64) DEFAULT SYS_GUID() PRIMARY KEY,"
            "  target_owner     VARCHAR2(128) NOT NULL,"
            "  objects_scanned  NUMBER,"
            "  facts_written    NUMBER,"
            "  notes            VARCHAR2(4000),"
            "  started_at       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,"
            "  finished_at      TIMESTAMP)"
        )
        print("created scan_history")
    except oracledb.DatabaseError as e:
        if e.args[0].code != 955:
            raise
        print("(already exists) scan_history")
agent_conn.commit()

## The Schema Scanner — Catalog Views as Training Data

Four scanners, each mining one catalog source into a `list[Fact]`. The simplest one is `_scan_tables` — your TODO. The other three (`_scan_columns`, `_scan_relationships`, `_scan_workload`) follow the same shape and are pre-built.

In [ ]:
@dataclass
class Fact:
    kind: str        # 'table' | 'column' | 'relationship' | 'query_pattern'
    subject: str     # e.g. 'SUPPLYCHAIN.VESSELS'
    body: str        # natural-language sentence the embedder will read
    metadata: dict   # owner, table, column, etc.

## Implement `_scan_tables`

> 📖 **See:** [Part 2 guide → TODO 1](../docs/part-2-oamp-memory.md#todo-1-implement-_scan_tables)

Mine `ALL_TABLES + ALL_TAB_COMMENTS` and emit one `Fact(kind="table")` per table. Build the `body` conditionally — skip parts that are `None`.


In [ ]:
# TODO 1: implement _scan_tables(conn, owner) -> list[Fact]
# Query: SELECT table_name, comments, num_rows, last_analyzed
#   FROM all_tables LEFT JOIN all_tab_comments USING (owner, table_name)
#   WHERE owner = :owner ORDER BY table_name
# For each row, build a body string and append a Fact(kind="table", subject=f"{owner}.{table}", ...).

def _scan_tables(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT t.table_name, tc.comments, t.num_rows, t.last_analyzed "
        "  FROM all_tables t "
        "  LEFT JOIN all_tab_comments tc "
        "    ON tc.owner = t.owner AND tc.table_name = t.table_name "
        " WHERE t.owner = :owner "
        " ORDER BY t.table_name"
    )
    facts: list[Fact] = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, comment, num_rows, last_analyzed in cur:
            body_parts = [f"Table {owner}.{table}."]
            if comment:
                body_parts.append(f"Documented purpose: {comment}")
            if num_rows is not None:
                body_parts.append(f"Approximate row count: {num_rows:,}.")
            if last_analyzed:
                body_parts.append(f"Statistics last gathered at {last_analyzed}.")
            facts.append(Fact(
                kind="table",
                subject=f"{owner}.{table}",
                body=" ".join(body_parts),
                metadata={"owner": owner, "table": table,
                          "num_rows": num_rows, "has_comment": bool(comment)},
            ))
    return facts

In [ ]:
# ✅ Checkpoint: TODO 1
assert callable(_scan_tables), "❌ TODO 1 incomplete — _scan_tables not defined"
# We can't fully test this until SUPPLYCHAIN exists (next cell), but at least the function shape is right.
import inspect as _ins
_sig = _ins.signature(_scan_tables)
assert list(_sig.parameters) == ["conn", "owner"], "❌ Wrong signature for _scan_tables"
print("✅ TODO 1 passed — _scan_tables defined with correct signature")

### Pre-built — the other three scanners

`_scan_columns`, `_scan_relationships`, `_scan_workload` follow the same shape as your `_scan_tables`. Read them once and notice how each catalog view becomes one English sentence the embedder can index.

In [ ]:
def _scan_columns(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT c.table_name, c.column_name, c.data_type, c.data_length, "
        "       c.nullable, cc.comments "
        "  FROM all_tab_columns c "
        "  LEFT JOIN all_col_comments cc "
        "    ON cc.owner = c.owner AND cc.table_name = c.table_name "
        "   AND cc.column_name = c.column_name "
        " WHERE c.owner = :owner "
        " ORDER BY c.table_name, c.column_id"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for table, col, dtype, dlen, nullable, comment in cur:
            dtype_str = dtype + (f"({dlen})" if dlen and dtype in ("VARCHAR2", "CHAR") else "")
            nullstr = "nullable" if nullable == "Y" else "NOT NULL"
            body = f"Column {owner}.{table}.{col} of type {dtype_str} ({nullstr})."
            if comment:
                body += f" Meaning: {comment}"
            facts.append(Fact(
                kind="column",
                subject=f"{owner}.{table}.{col}",
                body=body,
                metadata={"owner": owner, "table": table, "column": col,
                          "data_type": dtype, "nullable": nullable == "Y"},
            ))
    return facts


def _scan_relationships(conn, owner: str) -> list[Fact]:
    sql = (
        "SELECT c.constraint_name, c.table_name, acc.column_name, "
        "       rc.table_name AS r_table, rcc.column_name AS r_column "
        "  FROM all_constraints c "
        "  JOIN all_cons_columns acc "
        "    ON acc.owner = c.owner AND acc.constraint_name = c.constraint_name "
        "  JOIN all_constraints rc "
        "    ON rc.owner = c.r_owner AND rc.constraint_name = c.r_constraint_name "
        "  JOIN all_cons_columns rcc "
        "    ON rcc.owner = rc.owner AND rcc.constraint_name = rc.constraint_name "
        "   AND rcc.position = acc.position "
        " WHERE c.owner = :owner AND c.constraint_type = 'R'"
    )
    facts = []
    with conn.cursor() as cur:
        cur.execute(sql, owner=owner.upper())
        for cname, tbl, col, r_tbl, r_col in cur:
            facts.append(Fact(
                kind="relationship",
                subject=f"{owner}.{tbl}.{col}->{owner}.{r_tbl}.{r_col}",
                body=(f"Foreign key {cname}: {owner}.{tbl}.{col} references "
                      f"{owner}.{r_tbl}.{r_col}. Use this edge when joining the two tables."),
                metadata={"owner": owner, "child": tbl, "child_col": col,
                          "parent": r_tbl, "parent_col": r_col},
            ))
    return facts


def _scan_workload(conn, owner: str, limit: int = 50) -> list[Fact]:
    sql = (
        "SELECT /*+ FIRST_ROWS(:lim) */ "
        "       sql_id, sql_fulltext, executions, rows_processed "
        "  FROM v$sql "
        " WHERE parsing_schema_name = :owner "
        "   AND command_type IN (3, 6, 7, 189) "
        "   AND rownum <= :lim "
        " ORDER BY executions DESC NULLS LAST"
    )
    facts = []
    with conn.cursor() as cur:
        try:
            cur.execute(sql, owner=owner.upper(), lim=limit)
            for sql_id, text, execs, rows_proc in cur:
                if text is None:
                    continue
                stmt = (text.read() if hasattr(text, "read") else str(text)).strip()
                if len(stmt) > 2000:
                    stmt = stmt[:2000] + " /* ...truncated */"
                facts.append(Fact(
                    kind="query_pattern",
                    subject=f"v$sql:{sql_id}",
                    body=(f"A SQL statement observed in the workload for {owner} "
                          f"(executed {execs or 0} times, {rows_proc or 0} rows processed):\n{stmt}"),
                    metadata={"owner": owner, "sql_id": sql_id,
                              "executions": execs, "rows_processed": rows_proc},
                ))
        except oracledb.DatabaseError as e:
            print(f"  (workload scan skipped — V$SQL access: {e})")
    return facts


def scan_schema(conn, owner: str) -> list[Fact]:
    facts = []
    facts += _scan_tables(conn, owner)
    facts += _scan_columns(conn, owner)
    facts += _scan_relationships(conn, owner)
    facts += _scan_workload(conn, owner)
    return facts


print(f"scanners ready: _scan_tables (your TODO 1), _scan_columns, _scan_relationships, _scan_workload, scan_schema")

In [ ]:
# 2.3 — write_facts: idempotent upsert keyed on (kind, subject) with body_hash dedup.
# run_scan: scan a schema, persist facts, record bookkeeping in scan_history.

def _hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:32]


def write_facts(facts: list[Fact], scan_id: str | None = None) -> tuple[int, int, int]:
    new = updated = skipped = 0
    scan_id = scan_id or str(uuid.uuid4())
    store = memory_client._store

    for f in facts:
        body_hash = _hash(f.body)
        existing = store.list(
            "memory",
            user_id=USER_ID, agent_id=AGENT_ID,
            metadata_filter={"kind": f.kind, "subject": f.subject},
            limit=1,
        )
        meta = {**f.metadata, "kind": f.kind, "subject": f.subject,
                "body_hash": body_hash, "scan_id": scan_id}

        if not existing:
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            new += 1
        elif (existing[0].metadata or {}).get("body_hash") == body_hash:
            skipped += 1
        else:
            memory_client.delete_memory(existing[0].id)
            memory_client.add_memory(f.body, user_id=USER_ID, agent_id=AGENT_ID, metadata=meta)
            updated += 1
    return new, updated, skipped


def run_scan(conn, owner: str) -> dict:
    scan_id = str(uuid.uuid4())
    with conn.cursor() as cur:
        cur.execute("INSERT INTO scan_history (scan_id, target_owner, notes) VALUES (:id, :o, :n)",
                    id=scan_id, o=owner.upper(), n="started")
    conn.commit()
    facts = scan_schema(conn, owner)
    new, updated, skipped = write_facts(facts, scan_id=scan_id)
    with conn.cursor() as cur:
        cur.execute(
            "UPDATE scan_history SET objects_scanned = :n, facts_written = :w, "
            "  finished_at = CURRENT_TIMESTAMP, notes = :notes WHERE scan_id = :id",
            n=len(facts), w=new + updated,
            notes=f"new={new} updated={updated} skipped={skipped}",
            id=scan_id)
    conn.commit()
    return {"scan_id": scan_id, "facts_total": len(facts),
            "new": new, "updated": updated, "skipped": skipped}


print("write_facts + run_scan ready")

# Part 3 — Retrieval Strategies

> 📖 **Guide:** [`docs/part-3-retrieval.md`](../docs/part-3-retrieval.md)

> 🔧 **TODOs in this part (2):** **TODO 2** — `retrieve_knowledge`; **TODO 3** — `hybrid_rrf_search_memories`

This Part adds two retrieval surfaces over the OAMP store from Part 2:

1. **Vector search + cross-encoder rerank** — strong on meaning.
2. **Hybrid (vector + Oracle Text) fused via Reciprocal Rank Fusion** — strong on meaning *and* exact tokens.

Before we build them, the next cell seeds a realistic `SUPPLYCHAIN` schema (carriers, vessels, ports, voyages, containers, cargo) for the agent to reason over.

### Pre-built — seed the `SUPPLYCHAIN` demo schema

The cell below creates a `SUPPLYCHAIN` user, seven tables (carriers, ports, vessels, voyages, vessel_positions, containers, cargo_items), spatial metadata + indexes on the geometry columns, and ~570 rows of realistic data. Run it once.

Two columns to remember — these are the exact kind of facts a senior engineer remembers and an LLM hallucinates:

- **`vessels.capacity_teu`** is in 20-foot equivalent units (TEU), not tons.
- **`cargo_items.unit_value_cents`** is in USD **cents**, not dollars.

The scanner picks both up via `COMMENT ON COLUMN`.

In [ ]:
# Pivot: replace the toy DEMO schema with a deeper supply-chain dataset
# (carriers, vessels, ports, voyages, current positions, containers, cargo).
# Spatial columns (SDO_GEOMETRY) on `ports.location` and
# `vessel_positions.position`, with R-tree spatial indexes for SDO_WITHIN_DISTANCE
# queries. Same `DEMO_USER` Python variable so downstream cells don't have to
# move; the actual schema in Oracle is `SUPPLYCHAIN`.

DEMO_USER = "SUPPLYCHAIN"
DEMO_PASS = "SupplyPwd_2025"

with sys_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM all_users WHERE username = :u", u=DEMO_USER)
    if cur.fetchone()[0] == 0:
        cur.execute(f"CREATE USER {DEMO_USER} IDENTIFIED BY {DEMO_PASS}")
    cur.execute(f"GRANT CONNECT, RESOURCE, UNLIMITED TABLESPACE TO {DEMO_USER}")
    cur.execute(f"GRANT CREATE VIEW, CREATE PROCEDURE, CREATE TYPE TO {DEMO_USER}")
    # Spatial: every user that creates SDO_GEOMETRY columns needs this grant.
    try:
        cur.execute(f"GRANT EXECUTE ON MDSYS.SDO_GEOMETRY TO {DEMO_USER}")
    except oracledb.DatabaseError:
        pass
    cur.execute(f"GRANT SELECT ANY TABLE TO {AGENT_USER}")
sys_conn.commit()

demo_conn = connect(DEMO_USER, DEMO_PASS, SYS_DSN)


# ---------- Drop existing objects (idempotent re-run) -----------------------
DROP_VIEWS = ["voyage_dv", "vessel_dv"]
for v in DROP_VIEWS:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(f"DROP VIEW {v}")
    except oracledb.DatabaseError:
        pass

DROP_TABLES = ["cargo_items", "containers", "vessel_positions", "voyages",
               "vessels", "ports", "carriers"]
for t in DROP_TABLES:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(f"DROP TABLE {t} CASCADE CONSTRAINTS PURGE")
    except oracledb.DatabaseError:
        pass


# ---------- DDL --------------------------------------------------------------
DDL = [
    # Carriers
    """CREATE TABLE carriers (
         carrier_id      NUMBER(10) PRIMARY KEY,
         name            VARCHAR2(120) NOT NULL,
         hq_country      VARCHAR2(60),
         founded_year    NUMBER(4)
       )""",

    # Ports — has SDO_GEOMETRY plus denormalised lat/lon for duality view ergonomics
    """CREATE TABLE ports (
         port_code       VARCHAR2(5) PRIMARY KEY,
         name            VARCHAR2(120) NOT NULL,
         country         VARCHAR2(60),
         ocean_region    VARCHAR2(20) NOT NULL,
         terminal_count  NUMBER(3),
         latitude        NUMBER(10,6),
         longitude       NUMBER(10,6),
         location        SDO_GEOMETRY
       )""",

    # Vessels
    """CREATE TABLE vessels (
         vessel_id       NUMBER(10) PRIMARY KEY,
         carrier_id      NUMBER(10) NOT NULL REFERENCES carriers(carrier_id),
         name            VARCHAR2(120) NOT NULL,
         imo_number      VARCHAR2(10) UNIQUE,
         vessel_type     VARCHAR2(20) NOT NULL,
         capacity_teu    NUMBER(7),
         year_built      NUMBER(4),
         flag_country    VARCHAR2(60)
       )""",

    # Voyages — region (ocean) drives the §14.4 row policy.
    """CREATE TABLE voyages (
         voyage_id       NUMBER(10) PRIMARY KEY,
         vessel_id       NUMBER(10) NOT NULL REFERENCES vessels(vessel_id),
         origin_code     VARCHAR2(5) NOT NULL REFERENCES ports(port_code),
         dest_code       VARCHAR2(5) NOT NULL REFERENCES ports(port_code),
         departure_ts    TIMESTAMP,
         eta_ts          TIMESTAMP,
         status          VARCHAR2(20) NOT NULL,
         ocean_region    VARCHAR2(20) NOT NULL
       )""",

    # Current vessel positions — 1:1 with vessels for active fleet
    """CREATE TABLE vessel_positions (
         vessel_id       NUMBER(10) PRIMARY KEY REFERENCES vessels(vessel_id),
         position_ts     TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
         latitude        NUMBER(10,6),
         longitude       NUMBER(11,6),
         speed_knots     NUMBER(5,2),
         heading_deg     NUMBER(5,2),
         position        SDO_GEOMETRY
       )""",

    # Containers — what's loaded, who's shipping it
    """CREATE TABLE containers (
         container_id    NUMBER(10) PRIMARY KEY,
         voyage_id       NUMBER(10) NOT NULL REFERENCES voyages(voyage_id),
         container_no    VARCHAR2(11) UNIQUE NOT NULL,
         container_type  VARCHAR2(20) NOT NULL,
         consignor       VARCHAR2(120),
         consignee       VARCHAR2(120),
         status          VARCHAR2(20) NOT NULL
       )""",

    # Cargo line items per container
    """CREATE TABLE cargo_items (
         cargo_item_id   NUMBER(10) PRIMARY KEY,
         container_id    NUMBER(10) NOT NULL REFERENCES containers(container_id),
         hs_code         VARCHAR2(10),
         description     VARCHAR2(400),
         quantity        NUMBER(10),
         unit_value_cents NUMBER(15),
         weight_kg       NUMBER(10,2)
       )""",

    # Comments — the scanner picks these up as institutional knowledge
    "COMMENT ON TABLE carriers IS 'Shipping lines / freight carriers operating the fleet.'",
    "COMMENT ON TABLE vessels IS 'Individual ships owned/operated by carriers; identified by IMO number.'",
    "COMMENT ON COLUMN vessels.capacity_teu IS 'Cargo capacity in 20-foot equivalent units (TEU); never tons.'",
    "COMMENT ON TABLE ports IS 'Container ports worldwide. location is SDO_GEOMETRY (WGS84, SRID 8307); use SDO_WITHIN_DISTANCE for proximity queries.'",
    "COMMENT ON TABLE voyages IS 'A specific journey of a vessel between two ports. ocean_region = PACIFIC | ATLANTIC | INDIAN | MEDITERRANEAN drives DDS row policies.'",
    "COMMENT ON TABLE vessel_positions IS 'Current AIS-style position for each active vessel. position is SDO_GEOMETRY (WGS84).'",
    "COMMENT ON TABLE containers IS 'Container manifest: which container is on which voyage. status: LOADED, IN_TRANSIT, DISCHARGED, CUSTOMS_HOLD.'",
    "COMMENT ON TABLE cargo_items IS 'Line items inside each container; HS code, description, value (in cents — never dollars).'",
    "COMMENT ON COLUMN cargo_items.unit_value_cents IS 'Per-unit declared value in USD CENTS. Multiply by quantity for line total.'",
]

with demo_conn.cursor() as cur:
    for stmt in DDL:
        cur.execute(stmt)
demo_conn.commit()


# ---------- Spatial metadata + indexes --------------------------------------
SPATIAL = [
    # Wipe any stale metadata from a prior schema
    "DELETE FROM USER_SDO_GEOM_METADATA WHERE table_name IN ('PORTS', 'VESSEL_POSITIONS')",
    """INSERT INTO USER_SDO_GEOM_METADATA (table_name, column_name, diminfo, srid)
       VALUES ('PORTS', 'LOCATION',
               SDO_DIM_ARRAY(
                 SDO_DIM_ELEMENT('LON', -180, 180, 0.005),
                 SDO_DIM_ELEMENT('LAT',  -90,  90, 0.005)
               ),
               8307)""",
    """INSERT INTO USER_SDO_GEOM_METADATA (table_name, column_name, diminfo, srid)
       VALUES ('VESSEL_POSITIONS', 'POSITION',
               SDO_DIM_ARRAY(
                 SDO_DIM_ELEMENT('LON', -180, 180, 0.005),
                 SDO_DIM_ELEMENT('LAT',  -90,  90, 0.005)
               ),
               8307)""",
    "CREATE INDEX ports_loc_sx ON ports(location) INDEXTYPE IS MDSYS.SPATIAL_INDEX_V2",
    "CREATE INDEX vpos_loc_sx  ON vessel_positions(position) INDEXTYPE IS MDSYS.SPATIAL_INDEX_V2",
]

with demo_conn.cursor() as cur:
    for stmt in SPATIAL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            # Idempotent: index/metadata might already exist on re-run
            if e.args[0].code in (955, 1408, 13223, 13226, 29855):
                continue
            raise
demo_conn.commit()


# ---------- Seed data --------------------------------------------------------
import random
random.seed(42)  # determinism — same seed every run

# 15 carriers (real-world container shipping lines)
carriers = [
    (1,  "Maersk",         "Denmark",       1904),
    (2,  "MSC",            "Switzerland",   1970),
    (3,  "CMA CGM",        "France",        1978),
    (4,  "COSCO",          "China",         1961),
    (5,  "Hapag-Lloyd",    "Germany",       1970),
    (6,  "ONE",            "Japan",         2017),
    (7,  "Evergreen",      "Taiwan",        1968),
    (8,  "Yang Ming",      "Taiwan",        1972),
    (9,  "HMM",            "South Korea",   1976),
    (10, "ZIM",            "Israel",        1945),
    (11, "OOCL",           "Hong Kong",     1969),
    (12, "Wan Hai",        "Taiwan",        1965),
    (13, "PIL",            "Singapore",     1967),
    (14, "Matson",         "USA",           1882),
    (15, "ANL",            "Australia",     1956),
]

# 25 ports — mix of largest/strategic global ports with realistic coords
# (lon, lat) ordering matches SDO_POINT_TYPE(X=lon, Y=lat).
ports = [
    # code,  name,                country,       region,           term,  lat,        lon
    ("SGSIN", "Singapore",         "Singapore",   "INDIAN",         62,    1.264900,  103.832600),
    ("CNSHA", "Shanghai",          "China",       "PACIFIC",        43,   31.235400,  121.473700),
    ("CNSHK", "Shenzhen Yantian",  "China",       "PACIFIC",        20,   22.564200,  114.272100),
    ("CNNGB", "Ningbo-Zhoushan",   "China",       "PACIFIC",        38,   29.868400,  121.836700),
    ("HKHKG", "Hong Kong",         "Hong Kong",   "PACIFIC",        24,   22.319300,  114.169400),
    ("KRPUS", "Busan",             "South Korea", "PACIFIC",        21,   35.102800,  129.040300),
    ("AEJEA", "Jebel Ali (Dubai)", "UAE",         "INDIAN",         15,   25.012300,   55.061700),
    ("MYTPP", "Tanjung Pelepas",   "Malaysia",    "INDIAN",         11,    1.366000,  103.560000),
    ("LKCMB", "Colombo",           "Sri Lanka",   "INDIAN",          5,    6.951900,   79.851800),
    ("OMSLL", "Salalah",           "Oman",        "INDIAN",          4,   17.018300,   54.092400),
    ("USLAX", "Los Angeles",       "USA",         "PACIFIC",        25,   33.732500, -118.262000),
    ("USLGB", "Long Beach",        "USA",         "PACIFIC",        22,   33.760100, -118.214800),
    ("USNYC", "New York/Newark",   "USA",         "ATLANTIC",       16,   40.683800,  -74.137300),
    ("USSAV", "Savannah",          "USA",         "ATLANTIC",        9,   32.083200,  -81.097100),
    ("USHOU", "Houston",           "USA",         "ATLANTIC",       10,   29.730400,  -95.246700),
    ("NLRTM", "Rotterdam",         "Netherlands", "ATLANTIC",       30,   51.924400,    4.477700),
    ("DEHAM", "Hamburg",           "Germany",     "ATLANTIC",       17,   53.551100,    9.993700),
    ("BEANR", "Antwerp",           "Belgium",     "ATLANTIC",       21,   51.221300,    4.401800),
    ("FRLEH", "Le Havre",          "France",      "ATLANTIC",       12,   49.490000,    0.107400),
    ("GBFXT", "Felixstowe",        "UK",          "ATLANTIC",        8,   51.961100,    1.351200),
    ("ESVLC", "Valencia",          "Spain",       "MEDITERRANEAN",  10,   39.469900,   -0.376300),
    ("ESALG", "Algeciras",         "Spain",       "MEDITERRANEAN",   7,   36.131500,   -5.453300),
    ("ITGOA", "Genoa",             "Italy",       "MEDITERRANEAN",   8,   44.405600,    8.946300),
    ("EGSCB", "Suez Canal South",  "Egypt",       "MEDITERRANEAN",   1,   29.967600,   32.549400),
    ("JPYOK", "Yokohama / Tokyo",  "Japan",       "PACIFIC",        18,   35.443900,  139.638100),
]

# 30 vessels distributed across carriers
vessel_types = ["CONTAINER", "CONTAINER", "CONTAINER", "CONTAINER",  # bias container
                "BULK", "TANKER", "RORO"]
flag_country = ["Denmark", "Liberia", "Panama", "Singapore", "Marshall Islands",
                "Hong Kong", "Malta", "Bahamas", "Cyprus"]
def _imo():
    return f"IMO{random.randint(9000000, 9999999)}"

vessels = []
vessel_names_by_carrier = {
    1:  ["Maersk Madrid", "Maersk Edinburgh", "Maersk Hong Kong", "Maersk Tukang"],
    2:  ["MSC Oscar", "MSC Gulsun", "MSC Eloane", "MSC Mia"],
    3:  ["CMA CGM Marco Polo", "CMA CGM Alexander", "CMA CGM Jacques Saade"],
    4:  ["COSCO Shipping Universe", "COSCO Pisces", "COSCO Cancer"],
    5:  ["Hapag-Lloyd Berlin Express", "Hapag-Lloyd Hamburg Express"],
    6:  ["ONE Apus", "ONE Stork"],
    7:  ["Ever Ace", "Ever Given", "Ever Globe"],
    8:  ["YM Witness", "YM Wreath"],
    9:  ["HMM Algeciras", "HMM Oslo"],
    10: ["ZIM Mount Everest", "ZIM Kingston"],
    11: ["OOCL Hong Kong", "OOCL Spain"],
    12: ["Wan Hai 805"],
    13: ["Kota Pekarang"],
    14: ["Matson Daniel K. Inouye"],
    15: ["ANL Wahroonga"],
}
vid = 1
for cid, names in vessel_names_by_carrier.items():
    for name in names:
        vtype = "CONTAINER" if "Express" in name or "MSC" in name or "CMA" in name or "COSCO" in name or "OOCL" in name or "Maersk" in name or "HMM" in name or "ZIM" in name or "Ever" in name or "YM" in name or "ONE" in name or "Wan Hai" in name or "Matson" in name or "Kota" in name or "ANL" in name else random.choice(vessel_types)
        capacity = random.choice([5500, 8800, 11000, 14000, 18000, 20000, 23000, 24000])
        year = random.choice([2014, 2015, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])
        flag = random.choice(flag_country)
        vessels.append((vid, cid, name, _imo(), vtype, capacity, year, flag))
        vid += 1

# 60 voyages — one each per active route, status mix
voyage_statuses = ["ACTIVE"] * 30 + ["DELAYED"] * 10 + ["SCHEDULED"] * 10 + ["COMPLETED"] * 10

# Map ports to the ocean region a voyage between them will be tagged with.
# When origin/destination span regions, the voyage takes the *transit* region
# (the ocean it's sailing through) — close enough for the demo's row policy.
def _voyage_region(origin_port, dest_port):
    op_region = next(p[3] for p in ports if p[0] == origin_port)
    dp_region = next(p[3] for p in ports if p[0] == dest_port)
    if op_region == dp_region:
        return op_region
    # Cross-ocean — pick the dominant transit ocean
    pairs = {
        ("PACIFIC", "ATLANTIC"): "PACIFIC",   # Asia → US east via Panama
        ("ATLANTIC", "PACIFIC"): "ATLANTIC",
        ("INDIAN", "MEDITERRANEAN"): "INDIAN",
        ("MEDITERRANEAN", "INDIAN"): "MEDITERRANEAN",
        ("PACIFIC", "INDIAN"): "PACIFIC",
        ("INDIAN", "PACIFIC"): "INDIAN",
        ("ATLANTIC", "MEDITERRANEAN"): "ATLANTIC",
        ("MEDITERRANEAN", "ATLANTIC"): "MEDITERRANEAN",
        ("ATLANTIC", "INDIAN"): "INDIAN",
        ("INDIAN", "ATLANTIC"): "INDIAN",
        ("PACIFIC", "MEDITERRANEAN"): "PACIFIC",
        ("MEDITERRANEAN", "PACIFIC"): "MEDITERRANEAN",
    }
    return pairs.get((op_region, dp_region), op_region)

voyages = []
import datetime as _dt
now = _dt.datetime.utcnow()
random.shuffle(voyage_statuses)
for vid_i in range(60):
    voyage_id = vid_i + 1
    vessel_id = (vid_i % len(vessels)) + 1
    o = random.choice(ports)[0]
    d = random.choice([p for p in ports if p[0] != o])[0]
    status = voyage_statuses[vid_i]
    # Times relative to now
    if status == "SCHEDULED":
        dep = now + _dt.timedelta(days=random.randint(2, 30))
        eta = dep + _dt.timedelta(days=random.randint(7, 35))
    elif status == "COMPLETED":
        dep = now - _dt.timedelta(days=random.randint(40, 90))
        eta = dep + _dt.timedelta(days=random.randint(7, 35))
    elif status == "DELAYED":
        dep = now - _dt.timedelta(days=random.randint(10, 30))
        eta = now + _dt.timedelta(days=random.randint(3, 14))
    else:  # ACTIVE
        dep = now - _dt.timedelta(days=random.randint(2, 20))
        eta = now + _dt.timedelta(days=random.randint(5, 25))
    region = _voyage_region(o, d)
    voyages.append((voyage_id, vessel_id, o, d,
                    dep, eta, status, region))

# Vessel positions — one per ACTIVE/DELAYED voyage's vessel.
# Position is interpolated between origin and destination (rough midpoint with jitter).
ports_by_code = {p[0]: p for p in ports}
vessel_positions = []
seen_vessels = set()
for vy in voyages:
    voyage_id, vessel_id, o_code, d_code, dep, eta, status, region = vy
    if status not in ("ACTIVE", "DELAYED"):
        continue
    if vessel_id in seen_vessels:
        continue
    seen_vessels.add(vessel_id)
    op = ports_by_code[o_code]
    dp = ports_by_code[d_code]
    # Linear interpolation 30-70% of the way, with small lat/lon jitter
    t = random.uniform(0.30, 0.70)
    lat = op[5] + (dp[5] - op[5]) * t + random.uniform(-1.5, 1.5)
    lon = op[6] + (dp[6] - op[6]) * t + random.uniform(-1.5, 1.5)
    speed = round(random.uniform(12.0, 22.5), 2)
    heading = round(random.uniform(0, 360), 2)
    vessel_positions.append((vessel_id, lat, lon, speed, heading))

# Containers + cargo items
container_types = ["DRY", "DRY", "DRY", "REEFER", "HAZMAT", "OPEN_TOP", "DRY"]
container_statuses = ["LOADED", "IN_TRANSIT", "DISCHARGED", "CUSTOMS_HOLD"]
consignors = [
    "Foxconn Technology Group", "Bosch Automotive", "Samsung SDI",
    "Tesla Inc.", "Volkswagen AG", "Toyota Motor Corp.",
    "Lenovo Group", "Apple Inc.", "Nestle SA", "IKEA",
    "Maersk Logistics", "Caterpillar Inc.", "BASF SE",
    "Pfizer Inc.", "Glencore Agriculture",
]
consignees = [
    "Walmart Inc.", "Costco Wholesale", "Amazon.com Services",
    "Best Buy", "Home Depot", "Target Corp.",
    "Carrefour", "Tesco Stores", "Aldi Nord",
    "El Corte Ingles", "Loblaws", "Coles Group",
    "Sumitomo Electric Industries", "Pemex", "Petrobras",
]
cargo_catalog = [
    # (hs_code, description, base_value_cents, base_weight_kg, container_type)
    ("8517", "Smartphones, retail-packed", 28000, 0.18, "DRY"),
    ("8528", "LED televisions, 50-inch class", 32000, 18.5, "DRY"),
    ("8703", "Passenger automobiles", 2800000, 1500, "RORO"),
    ("8714", "Bicycle parts and accessories", 1200, 1.4, "DRY"),
    ("8419", "Industrial heat exchangers", 580000, 480, "OPEN_TOP"),
    ("3004", "Pharmaceuticals, packaged", 45000, 0.5, "DRY"),
    ("3004", "Pharmaceuticals, refrigerated", 89000, 0.5, "REEFER"),
    ("0303", "Frozen seafood, salmon", 4500, 1.0, "REEFER"),
    ("0810", "Fresh berries", 2800, 1.0, "REEFER"),
    ("0901", "Roasted coffee beans, bulk", 1800, 1.0, "DRY"),
    ("8506", "Lithium-ion battery cells (HAZ class 9)", 8500, 0.4, "HAZMAT"),
    ("3105", "Fertilizer, ammonium nitrate", 850, 25, "HAZMAT"),
    ("2710", "Petroleum lubricants, drums", 1900, 200, "HAZMAT"),
    ("9018", "Medical instruments and apparatus", 240000, 4.2, "DRY"),
    ("8471", "Personal computers and laptops", 95000, 2.3, "DRY"),
    ("9403", "Office furniture, knock-down", 12000, 35, "DRY"),
    ("4202", "Designer luggage and handbags", 35000, 1.8, "DRY"),
    ("6203", "Men's apparel, woven", 4500, 0.4, "DRY"),
    ("6204", "Women's apparel, woven", 5200, 0.4, "DRY"),
    ("8482", "Industrial ball bearings, assorted", 1900, 0.6, "DRY"),
    ("7308", "Steel structures, prefab", 32000, 200, "OPEN_TOP"),
    ("4011", "Pneumatic tires, passenger car", 9500, 8.5, "DRY"),
    ("8407", "Marine engines, spare parts", 165000, 75, "DRY"),
    ("8501", "Industrial electric motors", 92000, 110, "DRY"),
    ("3923", "Plastic articles for packing", 350, 0.05, "DRY"),
    ("4818", "Toilet paper and tissues, bulk", 280, 0.4, "DRY"),
    ("8504", "Power transformers", 145000, 320, "OPEN_TOP"),
    ("9405", "LED lighting fixtures", 5500, 1.2, "DRY"),
    ("0808", "Apples, fresh", 180, 0.18, "REEFER"),
    ("8703", "Electric vehicles, complete", 4200000, 2200, "RORO"),
]

containers = []
cargo_items = []
container_id = 1
cargo_id = 1
for v in voyages:
    voyage_id, vessel_id, _, _, _, _, status, _ = v
    if status == "COMPLETED":
        n_containers = random.randint(0, 2)
    elif status == "SCHEDULED":
        n_containers = random.randint(1, 3)
    else:
        n_containers = random.randint(2, 5)
    for _ in range(n_containers):
        ctype = random.choice(container_types)
        cstatus = random.choice(container_statuses)
        cno = f"MSCU{random.randint(1000000, 9999999)}"
        consignor = random.choice(consignors)
        consignee = random.choice(consignees)
        containers.append((container_id, voyage_id, cno, ctype, consignor, consignee, cstatus))
        # 1-3 cargo items per container, type-matched
        type_matched = [c for c in cargo_catalog if c[4] == ctype]
        pool = type_matched if type_matched else cargo_catalog
        for _ in range(random.randint(1, 3)):
            hs, desc, val, wt, _ctype = random.choice(pool)
            qty = random.choice([10, 25, 50, 100, 200, 500, 1000])
            cargo_items.append((cargo_id, container_id, hs, desc, qty,
                                int(val * random.uniform(0.85, 1.15)),
                                round(wt * qty, 2)))
            cargo_id += 1
        container_id += 1


# ---------- Inserts ---------------------------------------------------------
with demo_conn.cursor() as cur:
    cur.executemany(
        "INSERT INTO carriers (carrier_id, name, hq_country, founded_year) "
        "VALUES (:1, :2, :3, :4)",
        carriers,
    )

    # Named binds: SDO_POINT_TYPE references :lat/:lon a second time, and
    # the thin driver counts every reference for positional binds (DPY-4009).
    # Switching to dict-style named binds deduplicates by name.
    cur.executemany(
        "INSERT INTO ports (port_code, name, country, ocean_region, terminal_count, "
        "                   latitude, longitude, location) "
        "VALUES (:code, :name, :country, :region, :terminals, :lat, :lon, "
        "        SDO_GEOMETRY(2001, 8307, SDO_POINT_TYPE(:lon, :lat, NULL), NULL, NULL))",
        [
            {"code": p[0], "name": p[1], "country": p[2], "region": p[3],
             "terminals": p[4], "lat": p[5], "lon": p[6]}
            for p in ports
        ],
    )

    cur.executemany(
        "INSERT INTO vessels (vessel_id, carrier_id, name, imo_number, vessel_type, "
        "                     capacity_teu, year_built, flag_country) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7, :8)",
        vessels,
    )

    cur.executemany(
        "INSERT INTO voyages (voyage_id, vessel_id, origin_code, dest_code, "
        "                     departure_ts, eta_ts, status, ocean_region) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7, :8)",
        voyages,
    )

    cur.executemany(
        "INSERT INTO vessel_positions (vessel_id, latitude, longitude, "
        "                              speed_knots, heading_deg, position) "
        "VALUES (:vid, :lat, :lon, :speed, :heading, "
        "        SDO_GEOMETRY(2001, 8307, SDO_POINT_TYPE(:lon, :lat, NULL), NULL, NULL))",
        [
            {"vid": p[0], "lat": p[1], "lon": p[2], "speed": p[3], "heading": p[4]}
            for p in vessel_positions
        ],
    )

    cur.executemany(
        "INSERT INTO containers (container_id, voyage_id, container_no, container_type, "
        "                        consignor, consignee, status) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7)",
        containers,
    )

    cur.executemany(
        "INSERT INTO cargo_items (cargo_item_id, container_id, hs_code, description, "
        "                         quantity, unit_value_cents, weight_kg) "
        "VALUES (:1, :2, :3, :4, :5, :6, :7)",
        cargo_items,
    )

demo_conn.commit()

print(f"SUPPLYCHAIN seeded:")
print(f"  carriers:         {len(carriers):4d}")
print(f"  ports:            {len(ports):4d}  (with SDO_GEOMETRY + spatial index)")
print(f"  vessels:          {len(vessels):4d}")
print(f"  voyages:          {len(voyages):4d}")
print(f"  vessel_positions: {len(vessel_positions):4d}  (with SDO_GEOMETRY + spatial index)")
print(f"  containers:       {len(containers):4d}")
print(f"  cargo_items:      {len(cargo_items):4d}")


In [ ]:
# 3.1 — Run the scanner against SUPPLYCHAIN. After this cell, the OAMP store contains
# one memory per table, column, foreign-key, and observed query — all embedded and retrievable.

DEMO_USER = "SUPPLYCHAIN"
summary = run_scan(agent_conn, owner=DEMO_USER)
print(json.dumps(summary, indent=2))

In [ ]:
# 3.2 — Oracle Text index on the OAMP memory body. Required for hybrid retrieval (§3.4).
# SYNC (ON COMMIT) keeps the index fresh as new memories land.

TEXT_INDEX_NAME = "eda_memory_text_idx"
MEMORY_TABLE    = "eda_onnx_memory"

with agent_conn.cursor() as cur:
    try:
        cur.execute(f"DROP INDEX {TEXT_INDEX_NAME}")
        print(f"dropped existing {TEXT_INDEX_NAME}")
    except oracledb.DatabaseError as e:
        if e.args[0].code != 1418:
            raise
    try:
        cur.execute(
            f"CREATE INDEX {TEXT_INDEX_NAME} ON {MEMORY_TABLE}(content) "
            f"  INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')"
        )
        print(f"created Oracle Text index {TEXT_INDEX_NAME}")
    except oracledb.DatabaseError as e:
        if e.args[0].code == 29855:
            cur.execute(f"CREATE INDEX {TEXT_INDEX_NAME} ON {MEMORY_TABLE}(content) "
                        f"  INDEXTYPE IS CTXSYS.CONTEXT")
            print(f"created {TEXT_INDEX_NAME} (without SYNC ON COMMIT)")
        else:
            raise
agent_conn.commit()

## Implement `retrieve_knowledge`

> 📖 **See:** [Part 3 guide → TODO 2](../docs/part-3-retrieval.md#todo-2-implement-retrieve_knowledge)

`retrieve_knowledge(query, k, kinds=None)` is a two-stage call:

1. Cosine search via `memory_client.search` — oversample by 4× so the reranker has enough candidates.
2. Filter out `tool_output` memories so the log of past tool calls doesn't pollute knowledge retrieval.
3. Apply `kinds` filter if provided.
4. Rerank via `rerank(query, candidates, top_k=k, content_key="body")`.


In [ ]:
# TODO 2: implement retrieve_knowledge(query, k=5, kinds=None) -> list[dict]

def retrieve_knowledge(query: str, k: int = 5,
                       kinds: list[str] | None = None) -> list[dict]:
    """Semantic search over the agent's long-term memory."""
    # Defensive: LLMs sometimes pass `kinds` as a comma-separated string.
    if isinstance(kinds, str):
        kinds = [s.strip() for s in kinds.split(",") if s.strip()]
    if kinds == []:
        kinds = None

    cosine_fetch = k * 4
    hits = memory_client.search(
        query,
        user_id=USER_ID, agent_id=AGENT_ID,
        record_types=["memory"],
        max_results=cosine_fetch,
    )

    candidates: list[dict] = []
    for h in hits:
        meta = h.metadata or {}
        kind_value = meta.get("kind")
        if kind_value == "tool_output":
            continue
        if kinds is not None and (kind_value is None or kind_value not in kinds):
            continue
        candidates.append({
            "kind":     kind_value or "?",
            "subject":  meta.get("subject", ""),
            "body":     h.content,
            "metadata": meta,
            "distance": float(h.distance),
        })

    return rerank(query, candidates, top_k=k, content_key="body")

In [ ]:
# ✅ Checkpoint: TODO 2 — quick probe
hits = retrieve_knowledge("which table holds vessel capacity", k=3)
assert len(hits) > 0, "❌ TODO 2 returned no hits — check the scan ran and the function is correct."
for h in hits:
    print(f"  [{h['kind']:12s}] {h['subject'][:40]:40s}  {h['body'][:100]}")
print("\n✅ TODO 2 passed — retrieve_knowledge returns hits")

In [ ]:
# 3.3 — keyword_search_memories: full-text via Oracle Text. Pre-built.

def keyword_search_memories(query: str, k: int = 5) -> list[dict]:
    sql = f"""
        SELECT m.record_id, m.metadata,
               DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
               SCORE(1) AS score_txt
          FROM {MEMORY_TABLE} m
         WHERE CONTAINS(m.content, :kw, 1) > 0
           AND m.user_id  = :u AND m.agent_id = :a
           AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
         ORDER BY SCORE(1) DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        cur.execute(sql, kw=query, u=USER_ID, a=AGENT_ID, k=k)
        rows = []
        for record_id, metadata, content, score in cur:
            meta = metadata or {}
            if hasattr(content, "read"):
                content = content.read()
            rows.append({
                "record_id": record_id,
                "kind": meta.get("kind", "memory"),
                "subject": meta.get("subject", ""),
                "content": str(content or "")[:500],
                "score_txt": float(score) if score is not None else 0.0,
            })
    return rows


def hybrid_rrf_search_memories(query: str, k: int = 5,
                                per_list: int = 30, rrf_k: int = 60) -> list[dict]:
    """Hybrid retrieval over OAMP memories: vector + Oracle Text fused via RRF.
    Issued as one SQL statement — JOIN happens server-side."""
    sql = f"""
        WITH q_emb AS (
            SELECT TO_VECTOR(VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), 384, FLOAT64) AS emb
              FROM dual
        ),
        vec AS (
            SELECT m.record_id,
                   DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
                   m.metadata,
                   1 - VECTOR_DISTANCE(c.embedding, q_emb.emb, COSINE) AS sim_vec,
                   ROW_NUMBER() OVER (ORDER BY VECTOR_DISTANCE(c.embedding, q_emb.emb, COSINE)) AS r_vec
              FROM {MEMORY_TABLE} m
              JOIN eda_onnx_record_chunks c ON c.source_id = m.record_id
              CROSS JOIN q_emb
             WHERE m.user_id = :u AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY
        ),
        txt AS (
            SELECT m.record_id,
                   DBMS_LOB.SUBSTR(m.content, 4000, 1) AS content,
                   m.metadata,
                   SCORE(1) AS score_txt,
                   ROW_NUMBER() OVER (ORDER BY SCORE(1) DESC) AS r_txt
              FROM {MEMORY_TABLE} m
             WHERE CONTAINS(m.content, :kw, 1) > 0
               AND m.user_id = :u AND m.agent_id = :a
               AND (JSON_VALUE(m.metadata, '$.kind') IS NULL
                    OR JSON_VALUE(m.metadata, '$.kind') <> 'tool_output')
             FETCH FIRST :n ROWS ONLY
        ),
        fused AS (
            SELECT COALESCE(v.record_id, t.record_id) AS record_id,
                   COALESCE(v.content, t.content)     AS content,
                   COALESCE(v.metadata, t.metadata)   AS metadata,
                   NVL(v.sim_vec, 0)                  AS sim_vec,
                   NVL(t.score_txt, 0)                AS score_txt,
                   NVL(v.r_vec, 999999)               AS r_vec,
                   NVL(t.r_txt, 999999)               AS r_txt,
                   ( 1.0 / (:rrf_k + NVL(v.r_vec, 999999))
                   + 1.0 / (:rrf_k + NVL(t.r_txt, 999999)) ) AS rrf_score
              FROM vec v
              FULL OUTER JOIN txt t ON v.record_id = t.record_id
        )
        SELECT * FROM fused
         ORDER BY rrf_score DESC
         FETCH FIRST :k ROWS ONLY
    """
    with agent_conn.cursor() as cur:
        kw = f'"{query}"' if " " in query.strip() else query
        cur.execute(sql, q=query, kw=kw, u=USER_ID, a=AGENT_ID,
                    n=per_list, rrf_k=rrf_k, k=k)
        rows = []
        for rec_id, content, meta, sim_vec, score_txt, r_vec, r_txt, rrf in cur:
            if hasattr(content, "read"):
                content = content.read()
            rows.append({
                "record_id": rec_id,
                "kind": (meta or {}).get("kind", "memory"),
                "subject": (meta or {}).get("subject", ""),
                "content": str(content or "")[:500],
                "sim_vec":   float(sim_vec)   if sim_vec   is not None else 0.0,
                "score_txt": float(score_txt) if score_txt is not None else 0.0,
                "r_vec":     int(r_vec),
                "r_txt":     int(r_txt),
                "rrf_score": float(rrf),
            })
    return rows


print("keyword_search_memories + hybrid_rrf_search_memories ready")

## Run the three-way retrieval probe

> 📖 **See:** [Part 3 guide → Demo: three-way retrieval probe](../docs/part-3-retrieval.md#todo-5-run-the-three-way-retrieval-probe)

Same query through three retrievers — vector only, keyword only, hybrid via RRF. Watch the `r_vec` / `r_txt` columns in the hybrid output: rows that show up in **both** lists get the highest combined score.


In [ ]:
# Demo: three-way retrieval probe: run three retrievals on the same query and print results.

probe_q = "TEU 20-foot equivalent capacity unit for vessels"

print("=" * 80)
print(f"QUERY: {probe_q!r}")
print("=" * 80)

print("\n--- A) VECTOR ONLY (retrieve_knowledge) ---")
for h in retrieve_knowledge(probe_q, k=3):
    subject = (h.get("metadata") or {}).get("subject", "")
    print(f"  [{h.get('kind','memory'):12s}] {subject[:40]:40s}  {h['body'][:100]}")

print("\n--- B) KEYWORD ONLY (Oracle Text CONTAINS + SCORE) ---")
for h in keyword_search_memories(probe_q, k=3):
    print(f"  [{h['kind']:12s}] score={h['score_txt']:6.2f}  {h['subject'][:40]:40s}  {h['content'][:100]}")

print("\n--- C) HYBRID via RRF ---")
for h in hybrid_rrf_search_memories(probe_q, k=3):
    print(f"  rrf={h['rrf_score']:.4f}  r_vec={h['r_vec']:>3}  r_txt={h['r_txt']:>3}  "
          f"{h['subject'][:40]:40s}  {h['content'][:100]}")

# Part 4 — Tools & Skills

> 📖 **Guide:** [`docs/part-4-tools-and-skills.md`](../docs/part-6-tools-and-skills.md)

> 🔧 **TODO in this part:** **TODO 4** — register `tool_run_sql`

The agent needs to *do things*: execute SQL, run code, write to a scratchpad, fetch a document. In this harness:

- **Tools** live in a vector-indexed `toolbox` table; dispatched by the loop.
- **Skills** live in a vector-indexed `skillbox` table; surfaced as a manifest, body loaded on demand.

Both use the same in-database HNSW index. Vector retrieval keeps the per-turn prompt lean even when the registry grows past 30 tools.

![Toolbox flow](../images/cover-toolbox-flow.png)

In [ ]:
# 4.1 — DBFS scratchpad. A real filesystem inside the database (SecureFile LOBs).

import os.path

DBFS_TABLESPACE = "AGENT_DBFS_TS"
DBFS_STORE      = "AGENT_SCRATCH"
DBFS_MOUNT      = "/scratch"

with sys_conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM dba_tablespaces WHERE tablespace_name = :n",
                n=DBFS_TABLESPACE)
    if cur.fetchone()[0] == 0:
        cur.execute("SELECT name FROM v$datafile WHERE rownum = 1")
        sample_path = cur.fetchone()[0]
        dbfs_path = f"{os.path.dirname(sample_path)}/agent_dbfs01.dbf"
        cur.execute(f"CREATE TABLESPACE {DBFS_TABLESPACE} DATAFILE '{dbfs_path}' "
                    f"SIZE 100M AUTOEXTEND ON NEXT 50M MAXSIZE 2G")
        print(f"created tablespace {DBFS_TABLESPACE}")
    cur.execute(f"GRANT EXECUTE ON DBMS_DBFS_CONTENT TO {AGENT_USER}")
    cur.execute(f"GRANT EXECUTE ON DBMS_DBFS_SFS TO {AGENT_USER}")
    cur.execute(f"GRANT DBFS_ROLE TO {AGENT_USER}")
    cur.execute(f"ALTER USER {AGENT_USER} QUOTA UNLIMITED ON {DBFS_TABLESPACE}")
sys_conn.commit()

_DBFS_EXISTS_CODES = (955, 64007, 64008, 1)
_dbfs_stmts = [
    ("createfilesystem", f"BEGIN DBMS_DBFS_SFS.CREATEFILESYSTEM("
        f"  store_name => '{DBFS_STORE}', tbl_name => '{DBFS_STORE}_T', "
        f"  tbl_tbs => '{DBFS_TABLESPACE}', use_bf => FALSE); END;"),
    ("registerstore", f"BEGIN DBMS_DBFS_CONTENT.REGISTERSTORE("
        f"  store_name => '{DBFS_STORE}', provider_name => 'sample1', "
        f"  provider_package => 'DBMS_DBFS_SFS'); END;"),
    ("mountstore", f"BEGIN DBMS_DBFS_CONTENT.MOUNTSTORE("
        f"  store_name => '{DBFS_STORE}', store_mount => '{DBFS_MOUNT.lstrip('/')}'); END;"),
]
with agent_conn.cursor() as cur:
    for label, stmt in _dbfs_stmts:
        try:
            cur.execute(stmt)
            print(f"{label}: ok")
        except oracledb.DatabaseError as e:
            err_code = e.args[0].code if e.args else None
            if err_code in _DBFS_EXISTS_CODES or "already exists" in str(e).lower():
                print(f"{label}: (already exists)")
            else:
                print(f"{label}: {e}")
agent_conn.commit()

In [ ]:
class DBFS:
    """Minimal file-like wrapper over Oracle DBFS."""

    def __init__(self, conn, mount: str = DBFS_MOUNT):
        self.conn = conn
        self.mount = mount.rstrip("/")

    def _full(self, path: str) -> str:
        """Resolve to the absolute DBFS path. Accepts both bare and
        mount-qualified forms ('foo.sql' OR '/scratch/foo.sql') so
        callers don't accidentally double the prefix into
        '/scratch/scratch/foo.sql' (ORA-64001).
        """
        if not path.startswith("/"):
            path = "/" + path
        if path == self.mount or path.startswith(self.mount + "/"):
            return path
        return f"{self.mount}{path}"

    def write(self, path: str, content) -> None:
        """Create-or-overwrite the file at `path` under the scratchpad mount."""
        data = content.encode("utf-8") if isinstance(content, str) else content
        full = self._full(path)
        # DELETEFILE(path) is the path-based file delete in DBMS_DBFS_CONTENT;
        # the other arguments (filter, soft_delete, store_name, principal) all
        # default. The exception handler swallows "no such path" so the call
        # is a no-op when the file doesn't exist yet (i.e. "create or overwrite").
        delete_plsql = (
            "BEGIN "
            "  DBMS_DBFS_CONTENT.DELETEFILE(:p); "
            "EXCEPTION WHEN OTHERS THEN NULL; "
            "END;"
        )
        # CREATEFILE(path, properties, content) — properties and content are
        # IN/OUT, so they must be assignment-targets (locals), not literals.
        # We declare them inside the PL/SQL block and seed l_blob from the
        # bound bytes (forced to BLOB via setinputsizes below).
        create_plsql = (
            "DECLARE "
            "  l_props DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob  BLOB := :b; "
            "BEGIN "
            "  DBMS_DBFS_CONTENT.CREATEFILE("
            "    path       => :p,"
            "    properties => l_props,"
            "    content    => l_blob); "
            "  COMMIT; "
            "END;"
        )
        with self.conn.cursor() as cur:
            cur.execute(delete_plsql, p=full)
            cur.setinputsizes(b=oracledb.DB_TYPE_BLOB)
            cur.execute(create_plsql, p=full, b=data)
        self.conn.commit()

    def append(self, path: str, content) -> None:
        """Append `content` to `path`. If the file doesn't exist yet,
        behaves like `write`. Use for running findings logs that should
        grow without overwriting prior entries.
        """
        new = content.encode("utf-8") if isinstance(content, str) else content
        try:
            existing = self.read(path).encode("utf-8")
            sep = b"" if existing.endswith(b"\n") or not existing else b"\n"
            self.write(path, existing + sep + new)
        except FileNotFoundError:
            self.write(path, new)

    def read(self, path: str) -> str:
        full = self._full(path)
        # GETPATH(path, properties, content, item_type, ...) - first four are
        # mandatory in this version. We pass locals for all the OUT params and
        # only return content via :out.
        read_plsql = (
            "DECLARE "
            "  l_props     DBMS_DBFS_CONTENT_PROPERTIES_T := DBMS_DBFS_CONTENT_PROPERTIES_T(); "
            "  l_blob      BLOB; "
            "  l_item_type NUMBER; "
            "BEGIN "
            "  DBMS_DBFS_CONTENT.GETPATH("
            "    path       => :p,"
            "    properties => l_props,"
            "    content    => l_blob,"
            "    item_type  => l_item_type); "
            "  :out := l_blob; "
            "END;"
        )
        try:
            with self.conn.cursor() as cur:
                out = cur.var(oracledb.DB_TYPE_BLOB)
                cur.execute(read_plsql, p=full, out=out)
                blob = out.getvalue()
            if blob is None:
                raise FileNotFoundError(full)
            return blob.read().decode("utf-8", errors="replace")
        except oracledb.DatabaseError as e:
            # SFS provider raises ORA-64002 for non-existent paths instead
            # of returning NULL — translate so callers (esp. append) can
            # treat both the same way via FileNotFoundError.
            if e.args and e.args[0].code in (64002, 64007):
                raise FileNotFoundError(full) from e
            raise

    def list(self, path: str = "/") -> list[str]:
        """List every file under `path` in DBFS.

        Two strategies, in order:

        1. ``DBMS_DBFS_CONTENT.LIST(path, '*', 1)`` — the documented API. Works
           on some providers; the SecureFile (SFS) provider on Oracle Free 26ai
           raises ``ORA-64003`` ("unsupported operation") instead.

        2. Direct query on the SFS storage table (``AGENT_SCRATCH_T``).
           Important subtlety: SFS stores pathnames *without* the mount prefix
           (just '/comment.txt', not '/scratch/comment.txt'). We translate the
           caller's mount-qualified `full` path into an SFS-relative prefix
           for the LIKE filter, then prepend the mount back onto each result.
        """
        full = self._full(path)
        out: list[str] = []

        # Strategy 1 — DBMS_DBFS_CONTENT.LIST
        try:
            with self.conn.cursor() as cur:
                cur.execute(
                    "SELECT * FROM TABLE(DBMS_DBFS_CONTENT.LIST(:p, '*', 1))",
                    p=full)
                descs = cur.description or []
                path_idx = next(
                    (i for i, d in enumerate(descs) if "PATH" in (d[0] or "").upper()),
                    0,
                )
                for row in cur:
                    v = row[path_idx]
                    if v: out.append(v)
            if out:
                return out
        except oracledb.DatabaseError:
            pass

        # Strategy 2 — query the SFS storage table directly.
        store_table = f"{DBFS_STORE}_T"
        try:
            with self.conn.cursor() as cur:
                if full == self.mount or full == self.mount + "/":
                    sfs_prefix = "/"
                elif full.startswith(self.mount + "/"):
                    sfs_prefix = full[len(self.mount):]
                    if not sfs_prefix.endswith("/"):
                        sfs_prefix += "/"
                else:
                    sfs_prefix = "/"

                # pathtype = 1 -> file; std_deleted = 0 -> not tombstoned.
                cur.execute(
                    f"SELECT pathname FROM {store_table} "
                    f" WHERE pathtype = 1 AND std_deleted = 0 "
                    f"   AND pathname LIKE :p "
                    f" ORDER BY pathname",
                    p=f"{sfs_prefix}%")
                for (n,) in cur:
                    if n:
                        out.append(f"{self.mount}{n}")
        except oracledb.DatabaseError:
            pass

        return out


# Module-level instance every later cell uses — DBFS is just a thin wrapper
# around `agent_conn`, so it's safe to instantiate once at notebook scope.
scratch = DBFS(agent_conn)
print(f"DBFS scratch ready at {scratch.mount}")


### Pre-built — Oracle MLE sandbox

`exec_js(code)` runs a snippet of JavaScript inside Oracle's Multilingual Engine (MLE). Used for deterministic compute the LLM shouldn't do in its head — percentiles, weighted means, JSON reshaping. No subprocess, no GraalVM on your laptop, same trust boundary as `run_sql`.

In [ ]:
def exec_js(code: str) -> dict:
    """Evaluate a snippet of JavaScript inside Oracle MLE.

    Returns {"stdout": str, "stderr": str, "ok": bool}.
    """
    # Wrap user code so we can capture console.log output and route the
    # result back through mle-js-bindings (the EVAL `result` CLOB doesn't
    # auto-fill from an IIFE on this build, so we use the explicit binding
    # API and read it back via DBMS_MLE.IMPORT_FROM_MLE).
    wrapper = (
        '(function() {\n'
        '  let _stdout = "";\n'
        '  let _stderr = "";\n'
        '  let _ok = true;\n'
        '  const _origLog = console.log;\n'
        '  console.log = function() {\n'
        '    _stdout += Array.from(arguments).map(String).join(" ") + "\\n";\n'
        '  };\n'
        '  try {\n'
        + code + '\n'
        '  } catch (e) {\n'
        '    _stderr = String(e && e.message ? e.message : e) + "\\n" + (e && e.stack ? e.stack : "");\n'
        '    _ok = false;\n'
        '  } finally {\n'
        '    console.log = _origLog;\n'
        '  }\n'
        '  const bindings = require("mle-js-bindings");\n'
        '  bindings.exportValue("result", JSON.stringify({stdout: _stdout, stderr: _stderr, ok: _ok}));\n'
        '})();'
    )

    plsql = (
        "DECLARE "
        "  l_ctx    RAW(16); "
        "  l_result CLOB; "
        "BEGIN "
        "  l_ctx := DBMS_MLE.CREATE_CONTEXT(); "
        "  DBMS_MLE.EVAL(l_ctx, 'JAVASCRIPT', :source); "
        "  DBMS_MLE.IMPORT_FROM_MLE(l_ctx, 'result', l_result); "
        "  :result := l_result; "
        "  DBMS_MLE.DROP_CONTEXT(l_ctx); "
        "EXCEPTION WHEN OTHERS THEN "
        "  BEGIN DBMS_MLE.DROP_CONTEXT(l_ctx); EXCEPTION WHEN OTHERS THEN NULL; END; "
        "  RAISE; "
        "END;"
    )

    with agent_conn.cursor() as cur:
        out = cur.var(oracledb.DB_TYPE_CLOB)
        try:
            cur.setinputsizes(source=oracledb.DB_TYPE_CLOB)
            cur.execute(plsql, source=wrapper, result=out)
        except oracledb.DatabaseError as e:
            return {"stdout": "", "stderr": str(e), "ok": False}

    clob = out.getvalue()
    text = clob.read() if hasattr(clob, "read") else (clob or "")
    if not text:
        return {"stdout": "", "stderr": "MLE returned no output", "ok": False}
    try:
        return json.loads(text)
    except (ValueError, TypeError):
        return {"stdout": text, "stderr": "", "ok": True}


### Pre-built — toolbox DDL + `@register` decorator

The cell below creates the `toolbox` table with an HNSW vector index, defines the `@register` decorator, and provides `retrieve_tools(query, k)` for per-turn tool retrieval.

Read the `@register` body — it's the central abstraction of Part 4.

In [ ]:
import inspect
import re
import typing

TOOLS: dict[str, tuple] = {}            # name -> (callable, openai_schema)
ALWAYS_ON_TOOLS = {"search_knowledge", "run_sql", "remember", "exec_js"}
TOOL_EMBEDDING_DIM = ONNX_EMBED_DIM      # 384 for ALL_MINILM_L12_V2


# ---- toolbox table (DDL) -------------------------------------------------
_TOOLBOX_DDL = [
    (
        "CREATE TABLE toolbox ("
        "  name        VARCHAR2(128) PRIMARY KEY,"
        "  description CLOB NOT NULL,"
        "  parameters  JSON,"
        f" embedding   VECTOR({TOOL_EMBEDDING_DIM}, FLOAT32),"
        "  updated_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP"
        ")"
    ),
    (
        "CREATE VECTOR INDEX toolbox_emb_v ON toolbox(embedding) "
        "ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE"
    ),
]
with agent_conn.cursor() as cur:
    # If a previous run created `toolbox` with a different vector dim,
    # drop it cleanly so the new DDL takes.
    try:
        cur.execute("SELECT data_length FROM user_tab_columns "
                    " WHERE table_name = 'TOOLBOX' AND column_name = 'EMBEDDING'")
        row = cur.fetchone()
        if row and row[0] and row[0] not in (TOOL_EMBEDDING_DIM * 4, None):
            cur.execute("DROP TABLE toolbox PURGE")
            print(f"dropped toolbox (old vector dim) so we can rebuild at {TOOL_EMBEDDING_DIM}")
    except oracledb.DatabaseError:
        pass

    for stmt in _TOOLBOX_DDL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            code_ = e.args[0].code
            if code_ in (955, 1408):
                continue
            if code_ == 51962 and "VECTOR INDEX" in stmt.upper():
                print("!! ORA-51962: vector_memory_size is 0. HNSW index NOT created.")
                print("   The agent will fall back to linear cosine scan over the toolbox.")
                print("   Fix: run §3.2.1 to set vector_memory_size, `docker restart oracle-free`,")
                print("        restart the kernel, and re-run from §3.1.")
                continue
            raise

agent_conn.commit()


# ---- HNSW verification (loud, explicit) ----------------------------------
# We create the toolbox so the agent can do *vector retrieval* over its own
# tools at runtime - matters once you have 30+ tools and the per-turn prompt
# would otherwise carry every schema. The retrieval is only fast if there's
# an HNSW (Hierarchical Navigable Small World) graph index on `embedding`;
# without it, every turn does a full table scan with VECTOR_DISTANCE.
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT index_name, index_type "
        "  FROM user_indexes "
        " WHERE table_name = 'TOOLBOX' "
        "   AND index_type LIKE 'VECTOR%'"
    )
    vec_indexes = list(cur)

if vec_indexes:
    for name, kind in vec_indexes:
        print(f"OK: HNSW vector index in place -> {name} ({kind})")
else:
    print("!! HNSW vector index is MISSING on toolbox.embedding.")
    print("   Tool retrieval still works (linear cosine over a few rows is fine),")
    print("   but you'll lose the index benefit at scale. To enable: §3.2.1 + bounce + re-run §3.1+.")


# ---- type-hint -> JSON-schema helper -------------------------------------
_PRIMS = {int: "integer", float: "number", bool: "boolean", str: "string"}


def _hint_to_json(hint) -> dict:
    """Map a Python type hint into a JSON-schema fragment (best effort)."""
    origin = typing.get_origin(hint)
    if hint in _PRIMS:
        return {"type": _PRIMS[hint]}
    if origin in (list, typing.List):
        args = typing.get_args(hint) or (str,)
        return {"type": "array", "items": _hint_to_json(args[0])}
    if origin in (dict, typing.Dict):
        return {"type": "object"}
    if origin is typing.Union:
        non_none = [a for a in typing.get_args(hint) if a is not type(None)]
        if len(non_none) == 1:
            return _hint_to_json(non_none[0])
    return {"type": "string"}


def _build_schema(fn) -> tuple[str, str, dict, dict]:
    raw_name = fn.__name__
    name = raw_name[5:] if raw_name.startswith("tool_") else raw_name
    description = (inspect.getdoc(fn) or "").strip()
    if not description:
        raise ValueError(f"tool {name!r} has no docstring; @register needs one for retrieval")
    sig = inspect.signature(fn)
    hints = typing.get_type_hints(fn)
    properties: dict = {}
    required: list[str] = []
    for pname, param in sig.parameters.items():
        prop = _hint_to_json(hints.get(pname, str))
        if param.default is not inspect.Parameter.empty and param.default is not None:
            prop["default"] = param.default
        else:
            required.append(pname)
        properties[pname] = prop
    parameters = {"type": "object", "properties": properties, "required": required}
    openai_schema = {"type": "function",
                     "function": {"name": name,
                                  "description": description,
                                  "parameters": parameters}}
    return name, description, parameters, openai_schema


# ---- @register ------------------------------------------------------------
def register(fn):
    """Argument-less decorator: name from __name__, description from __doc__,
    parameters schema from the signature + type hints. Embedding for the
    `toolbox` row is computed in-DB via VECTOR_EMBEDDING - no Python-side
    embedder, no network call.
    """
    name, description, parameters, openai_schema = _build_schema(fn)
    TOOLS[name] = (fn, openai_schema)

    arg_text = " ".join(parameters["properties"].keys())
    embed_text = f"{name}: {description}\nargs: {arg_text}"

    with agent_conn.cursor() as cur:
        cur.execute(
            "MERGE INTO toolbox t USING (SELECT :tn AS n FROM dual) s ON (t.name = s.n) "
            "WHEN MATCHED THEN UPDATE SET description = :td, parameters = :tp, "
            f"                              embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
            "                              updated_at = CURRENT_TIMESTAMP "
            "WHEN NOT MATCHED THEN INSERT (name, description, parameters, embedding) "
            f"                       VALUES (:tn, :td, :tp, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA))",
            tn=name, td=description, tp=json.dumps(parameters), etext=embed_text)
    agent_conn.commit()
    return fn


# ---- retrieval ------------------------------------------------------------
def retrieve_tools(query: str, k: int = 6) -> list[dict]:
    """Top-k tool schemas for `query`, plus the always-on set.

    Stage 1: cosine search over `toolbox.embedding` (in-DB; HNSW when present).
    Stage 2: cross-encoder rerank via `rerank()` - in-DB when a reranker is
    loaded (§3.5), pass-through otherwise. Always-on tools are merged in last
    so they're guaranteed in the schema list regardless of rerank ordering.
    """
    # Stage 1: oversample so the reranker has signal to work with.
    cosine_fetch = k * 4
    rows: list[dict] = []
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, description FROM toolbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY",
            q=query, k=cosine_fetch)
        for name, desc in cur:
            desc_text = desc.read() if hasattr(desc, "read") else str(desc or "")
            rows.append({"name": name, "content": desc_text})

    # Stage 2: rerank by description against the user query.
    ranked = rerank(query, rows, top_k=k, content_key="content")

    schemas: dict[str, dict] = {}
    for r in ranked:
        if r["name"] in TOOLS:
            schemas[r["name"]] = TOOLS[r["name"]][1]

    # Always-on set merges in last - guaranteed inclusion.
    for name in ALWAYS_ON_TOOLS:
        if name in TOOLS:
            schemas[name] = TOOLS[name][1]
    return list(schemas.values())


## Register `tool_run_sql`

> 📖 **See:** [Part 4 guide → TODO 4](../docs/part-6-tools-and-skills.md#todo-4-register-tool_run_sql)

The agent's primary way to query live data. Must be **read-only** — `SELECT` and `WITH` only. Decorate with `@register`.


In [ ]:
# TODO 4: register tool_run_sql with @register.
# Requirements:
#   - signature: tool_run_sql(sql: str, max_rows: int = 50) -> str
#   - reject any statement that doesn't match _READ_ONLY (already defined)
#   - return JSON: {"columns": [...], "rows": [...], "row_count": N} or {"error": "..."}

_READ_ONLY = re.compile(r"^\s*(select|with)\b", re.IGNORECASE)


@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement (SELECT/WITH only) against the target Oracle AI Database
    and return up to `max_rows` rows as JSON. Reject any statement that isn't read-only.
    """
    if not _READ_ONLY.match(sql.strip()):
        return json.dumps({"error": "only SELECT / WITH statements are allowed in run_sql"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = []
            for i, r in enumerate(cur):
                if i >= max_rows:
                    break
                rows.append([(v.read() if hasattr(v, "read") else v) for v in r])
        return json.dumps({"columns": cols, "rows": rows, "row_count": len(rows)},
                          default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})

In [ ]:
# ✅ Checkpoint: TODO 4
assert "run_sql" in TOOLS, "❌ TODO 4 incomplete — tool_run_sql not registered"
out = json.loads(tool_run_sql("SELECT 1 AS one FROM dual"))
assert out.get("row_count") == 1, "❌ tool_run_sql didn't execute correctly"
out_bad = json.loads(tool_run_sql("DROP TABLE x"))
assert "error" in out_bad, "❌ tool_run_sql allowed a non-SELECT statement"
print("✅ TODO 4 passed — tool_run_sql registered and read-only enforced")

### Pre-built — the rest of the toolset

`scan_database`, `search_knowledge`, `exec_js`, `scratch_*`, `remember`, `load_skill`, `list_skills`.

In [ ]:
# ============== Tool implementations =====================================

@register
def tool_scan_database(owner: str) -> str:
    """Scan the specified schema of the target Oracle AI Database and update institutional knowledge.
    Run this when the user asks about a schema you have never seen or when you suspect the knowledge store is stale.
    `owner` is the schema owner (e.g. 'DEMO').
    """
    return json.dumps(run_scan(agent_conn, owner=owner))


@register
def tool_search_knowledge(query: str, k: int = 5, kinds: list[str] | None = None) -> str:
    """Search institutional knowledge (what the agent has learned about the target database) by semantic similarity.
    Use this BEFORE running SQL to discover which tables and columns are relevant.
    `kinds` is an optional filter: table, column, relationship, query_pattern, correction.
    """
    hits = retrieve_knowledge(query, k=k, kinds=kinds)
    for h in hits:
        h["body"] = h["body"][:500]
    return json.dumps(hits)


_READ_ONLY = re.compile(r"^\s*(select|with)\b", re.IGNORECASE)


@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement (SELECT/WITH only) against the target Oracle AI Database
    and return up to `max_rows` rows as JSON. Reject any statement that isn't read-only.
    """
    if not _READ_ONLY.match(sql.strip()):
        return json.dumps({"error": "only SELECT / WITH statements are allowed in run_sql"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = []
            for i, r in enumerate(cur):
                if i >= max_rows: break
                rows.append([(v.read() if hasattr(v, "read") else v) for v in r])
        return json.dumps({"columns": cols, "rows": rows, "row_count": len(rows)},
                          default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})


@register
def tool_exec_js(code: str) -> str:
    """Execute JavaScript inside Oracle MLE (no filesystem, no network).
    Good for arithmetic, string formatting, JSON reshaping, simple aggregations.
    `console.log(...)` output comes back as `stdout`.
    """
    return json.dumps(exec_js(code))


@register
def tool_scratch_write(path: str, content: str) -> str:
    """Write a short-term note to the DBFS scratchpad.
    Use for draft SQL, intermediate results, running calculations you want to reference later in the same turn.
    """
    scratch.write(path, content)
    return json.dumps({"ok": True, "path": path, "bytes": len(content)})


@register
def tool_scratch_append(path: str, content: str) -> str:
    """Append text to the end of a DBFS scratchpad file (or create it).
    Use this instead of `scratch_write` when you want to ADD to a running
    log without losing prior entries — e.g. /scratch/findings.md as you
    discover facts across multiple turns, /scratch/transcript.md, etc.
    For SQL drafts or "latest is truth" content, prefer scratch_write.
    """
    scratch.append(path, content)
    return json.dumps({"ok": True, "path": path, "appended_bytes": len(content)})


@register
def tool_scratch_read(path: str) -> str:
    """Read from the DBFS scratchpad."""
    try:
        return json.dumps({"content": scratch.read(path)})
    except FileNotFoundError:
        return json.dumps({"error": f"not found: {path}"})


@register
def tool_remember(subject: str, body: str, kind: str = "correction") -> str:
    """Persist a correction or learning into institutional knowledge so future turns benefit.
    Use when the user corrects you, or when you discover a non-obvious fact that future retrievals should surface.
    `subject` is a short label (e.g. 'SALES.ORDERS.total_cents'); `body` is the fact written as a full sentence.
    """
    fact = Fact(kind=kind, subject=subject, body=body, metadata={"source": "agent_remember"})
    new, updated, _ = write_facts([fact])
    return json.dumps({"ok": True, "new": new, "updated": updated})


print(f"registered {len(TOOLS)} tools: {sorted(TOOLS)}")


### Pre-built — skillbox table, ingestion, and `load_skill` / `list_skills` tools

The skillbox is seeded from [`oracle/skills`](https://github.com/oracle/skills) — ~155 markdown playbooks across 17 categories. Top-3 names + descriptions are surfaced into every prompt as a manifest; the full body is loaded on demand via `load_skill(name)`.

![Skillbox flow](../images/cover-skillbox-flow.png)

In [ ]:
SKILLBOX_DDL = [
    """
    CREATE TABLE skillbox (
      name        VARCHAR2(160) PRIMARY KEY,
      category    VARCHAR2(64),
      description VARCHAR2(2000) NOT NULL,
      body        CLOB NOT NULL,
      source_url  VARCHAR2(2000),
      source_sha  VARCHAR2(64),
      embedding   VECTOR(384, FLOAT32),
      metadata    JSON,
      updated_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """,
    """
    CREATE VECTOR INDEX skillbox_emb_v ON skillbox(embedding)
      ORGANIZATION INMEMORY NEIGHBOR GRAPH DISTANCE COSINE
    """,
]

with agent_conn.cursor() as cur:
    for stmt in SKILLBOX_DDL:
        try:
            cur.execute(stmt)
        except oracledb.DatabaseError as e:
            code_ = e.args[0].code
            if code_ in (955, 1408):  # already exists
                continue
            if code_ == 51962 and "VECTOR INDEX" in stmt.upper():
                print("!! ORA-51962: vector_memory_size = 0 — HNSW index on skillbox NOT created.")
                print("   See §3.2.1; skillbox vector retrieval will linear-scan until you bounce.")
                continue
            raise
agent_conn.commit()

# Loud verification (mirrors §10/cell 73's pattern)
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT index_name, index_type FROM user_indexes "
        " WHERE table_name = 'SKILLBOX' AND index_type LIKE 'VECTOR%'"
    )
    vec = list(cur)
if vec:
    for n, t in vec:
        print(f"OK: HNSW vector index in place -> {n} ({t})")
else:
    print("!! HNSW vector index missing on skillbox.embedding.")


In [ ]:
import re as _re_skill

def _parse_skill_md(text: str, fallback_name: str) -> str:
    """Pull a one-paragraph description out of a markdown skill file.

    Looks for the first non-empty paragraph after the H1 (and after any
    horizontal-rule frontmatter delimiters). Falls back to fallback_name when
    nothing useful is in the body.
    """
    lines = text.splitlines()
    n, i = len(lines), 0

    # Skip any YAML frontmatter ("---" ... "---") at the top.
    if i < n and lines[i].strip() == "---":
        i += 1
        while i < n and lines[i].strip() != "---":
            i += 1
        i += 1  # skip closing ---

    # Skip leading blank lines.
    while i < n and not lines[i].strip():
        i += 1
    # Skip the H1 line if present.
    if i < n and lines[i].lstrip().startswith("# "):
        i += 1
    # Skip blank lines after H1.
    while i < n and not lines[i].strip():
        i += 1
    # Collect first paragraph (until a blank line or another heading).
    para = []
    while i < n and lines[i].strip() and not lines[i].lstrip().startswith("#"):
        para.append(lines[i].strip())
        i += 1

    description = " ".join(para).strip()
    if not description:
        description = fallback_name
    if len(description) > 1800:
        description = description[:1797].rsplit(" ", 1)[0] + "..."
    return description


# Smoke test on a synthetic markdown blob.
_test = """# Schema Discovery

Before composing a SQL query against an unfamiliar table, an agent should mine
the catalog views (ALL_TAB_COLUMNS, ALL_CONS_COLUMNS) to confirm column names
and key relationships.

## Steps

1. Query ALL_TABLES.
"""
print(_parse_skill_md(_test, fallback_name="agent/schema-discovery"))


In [ ]:
import urllib.request as _urlreq
import tarfile as _tarfile
import io as _io
import hashlib as _hashlib
import json as _json_skill
import datetime as _dt_skill

ORACLE_SKILLS_REPO = "oracle/skills"
ORACLE_SKILLS_BASE = "db"
SKILLS_TARBALL_URL = f"https://api.github.com/repos/{ORACLE_SKILLS_REPO}/tarball/main"


def _download_repo_tarball(url: str = SKILLS_TARBALL_URL) -> dict[str, bytes]:
    """One unauthenticated request — returns {repo_relative_path: bytes} for every
    .md file under db/. Avoids the 60-req/hr GitHub API limit on per-file fetches.
    """
    req = _urlreq.Request(url, headers={"User-Agent": "eda-skillbox-ingester"})
    with _urlreq.urlopen(req, timeout=60) as resp:
        data = resp.read()

    files: dict[str, bytes] = {}
    with _tarfile.open(fileobj=_io.BytesIO(data), mode="r:gz") as tar:
        for member in tar:
            if not member.isfile():
                continue
            # Tarball paths look like "oracle-skills-<sha7>/db/agent/schema-discovery.md".
            parts = member.name.split("/", 1)
            if len(parts) < 2:
                continue
            rel = parts[1]
            if not rel.startswith(f"{ORACLE_SKILLS_BASE}/") or not rel.endswith(".md"):
                continue
            f = tar.extractfile(member)
            if f is not None:
                files[rel] = f.read()
    return files


def ingest_skills_from_oracle(verbose: bool = True) -> dict:
    """Pull every db/<category>/<skill>.md from oracle/skills and MERGE into skillbox.

    Idempotent: rows with unchanged source_sha are skipped. Run repeatedly without
    side effects.
    """
    if verbose:
        print(f"downloading tarball from {SKILLS_TARBALL_URL} ...")
    raw_files = _download_repo_tarball()
    if verbose:
        print(f"  fetched {len(raw_files)} .md files under db/")

    new = updated = skipped = 0
    skipped_overview = 0

    for rel_path, body_bytes in sorted(raw_files.items()):
        # rel_path = "db/<category>/<file>.md"
        parts = rel_path.split("/")
        if len(parts) != 3:
            # nested deeper than db/<cat>/file.md; skip — the repo isn't deeply nested
            # but be defensive in case future versions add subfolders
            continue
        _, category, filename = parts
        # Skip category-level overview file; only mirror leaf skills
        if filename == "SKILL.md":
            skipped_overview += 1
            continue

        body = body_bytes.decode("utf-8", errors="replace")
        file_stem = filename[:-3]  # drop ".md"
        full_name = f"{category}/{file_stem}"
        sha = _hashlib.sha256(body.encode("utf-8")).hexdigest()[:32]

        # Idempotency: skip if same SHA already in DB
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT source_sha FROM skillbox WHERE name = :n",
                n=full_name,
            )
            row = cur.fetchone()
        if row and row[0] == sha:
            skipped += 1
            continue

        description = _parse_skill_md(body, fallback_name=full_name)
        embed_text = f"{full_name}\n{description}"
        source_url = (
            f"https://raw.githubusercontent.com/{ORACLE_SKILLS_REPO}/main/{rel_path}"
        )
        meta = _json_skill.dumps({
            "category": category,
            "ingested_at": _dt_skill.datetime.now(_dt_skill.timezone.utc).isoformat(timespec="seconds"),
            "source_repo": ORACLE_SKILLS_REPO,
            "rel_path": rel_path,
        })

        with agent_conn.cursor() as cur:
            # body is CLOB and skill files routinely exceed VARCHAR2 limits;
            # metadata is JSON. Bind both with explicit types so the thin driver
            # doesn't try to send them as oversized VARCHAR2 payloads (ORA-03146).
            cur.setinputsizes(body=oracledb.DB_TYPE_CLOB, md=oracledb.DB_TYPE_JSON)
            cur.execute(
                "MERGE INTO skillbox t "
                "USING (SELECT :n AS name FROM dual) s ON (t.name = s.name) "
                "WHEN MATCHED THEN UPDATE SET "
                "  category = :cat, "
                "  description = :dsc, "
                "  body = :body, "
                "  source_url = :url, "
                "  source_sha = :sha, "
                f" embedding = VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), "
                "  metadata = :md, "
                "  updated_at = CURRENT_TIMESTAMP "
                "WHEN NOT MATCHED THEN INSERT "
                "  (name, category, description, body, source_url, source_sha, embedding, metadata) "
                "  VALUES (:n, :cat, :dsc, :body, :url, :sha, "
                f"          VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :etext AS DATA), :md)",
                n=full_name, cat=category, dsc=description, body=body,
                url=source_url, sha=sha, etext=embed_text, md=meta,
            )
            if row is None:
                new += 1
                marker = "+"
            else:
                updated += 1
                marker = "~"
        agent_conn.commit()
        if verbose:
            print(f"  {marker} {full_name}")

    return {
        "new": new,
        "updated": updated,
        "skipped": skipped,
        "skipped_overview": skipped_overview,
        "total_in_repo": len(raw_files),
    }

# Run the ingest. Idempotent — re-runs MERGE on changed source_sha and skip
# unchanged files, so this is safe to leave at the bottom of the cell.
_skill_ingest_result = ingest_skills_from_oracle(verbose=False)
print(f"skillbox ingest: {_skill_ingest_result}")


In [ ]:
@register
def tool_load_skill(name: str) -> str:
    """Load the full content of a named skill from the skillbox.
    Use this when the system prompt's "Available skills" manifest lists a skill
    relevant to the current task. The full markdown guide is returned and you
    should follow its instructions for the duration of the task.
    `name` is the full namespace, e.g. "agent/schema-discovery".
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT description, body, source_url, category "
            "  FROM skillbox WHERE name = :n",
            n=name,
        )
        row = cur.fetchone()
    if not row:
        return json.dumps({"error": f"no skill named {name!r}; call list_skills(query) to find available skills"})
    desc, body, url, category = row
    body_text = body.read() if hasattr(body, "read") else str(body or "")
    return json.dumps({
        "name": name,
        "category": category,
        "description": desc,
        "source_url": url,
        "body": body_text,
    })


@register
def tool_list_skills(query: str, k: int = 5) -> str:
    """Search the skillbox semantically. Returns top-k skills (name + description).
    Use when the system prompt's manifest didn't surface the right skill for the
    current task. Then call load_skill(name) on the most relevant one.
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, category, description FROM skillbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY",
            q=query, k=k,
        )
        hits = [{"name": n, "category": c, "description": d} for n, c, d in cur]
    return json.dumps(hits)


# load_skill is always-on so the manifest in the system prompt is never a dead reference.
ALWAYS_ON_TOOLS.add("load_skill")
print(f"registered: load_skill, list_skills  (TOOLS total: {len(TOOLS)}; always-on: {sorted(ALWAYS_ON_TOOLS)})")


In [ ]:
def build_skill_manifest(query: str, k: int = 3) -> str:
    """Top-k skills relevant to `query` formatted as a one-line-per-skill manifest.
    Returns an empty string when the skillbox is empty (graceful degradation —
    the rest of the prompt is unaffected).

    `build_context` (defined in §11) calls this directly, so the agent loop's
    user message starts with the manifest whenever §11.5 has been run.
    """
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT name, description FROM skillbox "
                f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
                " FETCH FIRST :k ROWS ONLY",
                q=query, k=k,
            )
            rows = list(cur)
    except oracledb.DatabaseError:
        return ""
    if not rows:
        return ""
    lines = [f"  - {n} — {(d.read() if hasattr(d, 'read') else d)[:240]}" for n, d in rows]
    return (
        "Available skills (call load_skill(name) to read the full guide and follow it):\n"
        + "\n".join(lines) + "\n\n"
    )


# Part 5 — The Agent Loop

> 📖 **Guide:** [`docs/part-5-agent-loop.md`](../docs/part-7-agent-loop.md)

> 🔧 **TODO in this part:** **TODO 5** — `agent_turn`

Everything we've built so far is plumbing. **This** is the agent.

```
build_context → call LLM with retrieved tools → if tool_calls: dispatch  else: final → log
```

Every turn assembles a context block from OAMP + retrieved schema facts, calls the chat LLM with the top-k tools surfaced from the toolbox, and either dispatches the tool calls the model emitted or breaks out with the model's natural-language answer.

In [ ]:
SYSTEM_PROMPT = (
    "You are an Enterprise Data Agent operating against an Oracle AI Database.\n"
    "\n"
    "Your job is to answer natural-language questions about the data by reasoning over:\n"
    "  - the institutional knowledge you have accumulated about the database\n"
    "    (tables, columns, relationships, observed query patterns, past corrections,\n"
    "    AND episodic memories of prior conversations on this or other threads);\n"
    "  - the live database, via read-only SQL when you need runtime facts;\n"
    "  - a scratchpad for intermediate notes;\n"
    "  - an in-database JavaScript sandbox via Oracle MLE (`exec_js`) for computation.\n"
    "\n"
    "How to work:\n"
    "  1. ALWAYS call `search_knowledge` first with a paraphrase of the user's question.\n"
    "     Use the results to pick the right tables before writing any SQL.\n"
    "  2. If the user references a schema you have no facts about, call `scan_database`\n"
    "     on that owner to build institutional knowledge, THEN search again.\n"
    "  3. Keep SQL read-only. Use `run_sql` only; never attempt DDL or DML through it.\n"
    "  4. For numeric work over fetched rows - mean / median / percentile / max /\n"
    "     custom aggregation / unit conversion / formatting - you MUST fetch the raw\n"
    "     values with `run_sql` and then call `exec_js` to compute. Do not compute\n"
    "     statistics in your head, and do not lean only on SQL aggregation when JS is\n"
    "     more honest about the math (e.g. percentiles, weighted means, post-fetch\n"
    "     reshaping). The MLE sandbox runs inside Oracle - same trust boundary as\n"
    "     `run_sql`, no network egress, no separate install - so prefer it over\n"
    "     in-head arithmetic every time.\n"
    "  5. For non-trivial SQL (more than ~3 lines, or involving multiple tables /\n"
    "     joins), draft it first to the DBFS scratchpad via `scratch_write` to a\n"
    "     path like `/scratch/<task>.sql`, then `scratch_read` it before passing to\n"
    "     `run_sql`. The scratchpad is a real file system inside Oracle - files\n"
    "     persist across tool calls AND across turns on the same thread.\n"
    "     - `scratch_write(path, content)` REPLACES the file. Use for SQL drafts,\n"
    "       plan revisions, anything where 'latest is the truth'.\n"
    "     - `scratch_append(path, content)` ADDS to the end. Use for running\n"
    "       findings logs (`/scratch/findings.md`), transcripts, anything you\n"
    "       want to grow across multiple steps or turns without overwriting.\n"
    "       BATCH your appends: one `scratch_append` per row of data is wasteful\n"
    "       and burns the iteration budget. Combine many rows into ONE call.\n"
    "  6. When you discover a non-obvious fact, or the user corrects you, call `remember`\n"
    "     so future turns benefit.\n"
    "  7. Prefer short, direct answers. Quote table and column names verbatim.\n"
    "  8. If a tool fails, read the error, adjust, and try once more - don't spin.\n"
    "\n"
    "Never fabricate a table or column. If you're unsure, say so and propose a scan."
)


In [ ]:
import concurrent.futures

# Cache of OAMP thread objects keyed by the harness-level thread_id.
THREADS: dict[str, object] = {}


def get_thread(thread_id: str):
    """Look up or create the OAMP thread for this harness-level thread_id."""
    if thread_id in THREADS:
        return THREADS[thread_id]
    try:
        thread = memory_client.get_thread(thread_id)
    except Exception:
        thread = memory_client.create_thread(
            thread_id=thread_id,
            user_id=USER_ID,
            agent_id=AGENT_ID,
            memory_extraction_frequency=2,
            memory_extraction_window=4,
            enable_context_summary=True,
            context_summary_update_frequency=4,
        )
    THREADS[thread_id] = thread
    return thread


def build_context(thread_id: str, user_query: str, k_knowledge: int = 3) -> str:
    """Assemble the prompt context block for one user turn.

    Three layers stack into one user message:
      1. Skill manifest (top-3 from §11.5's `skillbox`, only if defined yet —
         graceful fallback when the cell hasn't been run).
      2. OAMP context card — relevant memories + recent turns + running summary.
      3. Top-k schema fact memories matching the current question.

    `k_knowledge=3` keeps the per-call prompt small enough that LLM round-trip
    latency stays under the agent_turn budget. Bump it for richer context
    once you've wired prompt caching.
    """
    parts: list[str] = []

    # Layer 1 — skill manifest (defined in §11.5 above). The `try` keeps Part 11
    # runnable even if §11.5 hasn't been executed yet on this kernel.
    try:
        manifest = build_skill_manifest(user_query, k=3)
    except NameError:
        manifest = ""
    if manifest:
        parts.append(manifest.rstrip())

    thread = get_thread(thread_id)

    # Layer 2 — OAMP context card
    card = thread.get_context_card()
    card_text = str(card) if card else ""
    if card_text:
        parts.append("## Memory context (from OAMP)")
        parts.append(card_text)

    # Layer 3 — institutional knowledge top-k
    hits = retrieve_knowledge(user_query, k=k_knowledge)
    if hits:
        parts.append("\n## Institutional knowledge (top matches)")
        for h in hits:
            parts.append(f"- ({h['kind']}) {h['subject']} - {h['body'][:280]}")

    parts.append("\n## User question")
    parts.append(user_query)
    return "\n".join(parts)


# Fire-and-forget logger. OAMP's `add_messages` runs synchronous
# extraction-LLM work when the message-extraction window fills — that's a
# 10–30 s LLM round-trip we don't want to block the agent loop on. We
# submit the write to a small pool and return immediately; any error
# surfaces on the next pool flush via `_drain_log_errors`. Persistence is
# best-effort by design — losing a log line is recoverable; blocking the
# loop on every turn is not.
_LOG_EXECUTOR = concurrent.futures.ThreadPoolExecutor(
    max_workers=4, thread_name_prefix="oamp-log")
_LOG_PENDING: list[concurrent.futures.Future] = []


def _drain_log_errors() -> None:
    """Pop any completed log futures and surface their errors. Cheap;
    runs synchronously but only over already-finished work."""
    still_pending = []
    for fut in _LOG_PENDING:
        if not fut.done():
            still_pending.append(fut)
            continue
        exc = fut.exception()
        if exc is not None:
            print(f"  ! background log_message failed: {type(exc).__name__}: {exc}")
    _LOG_PENDING[:] = still_pending


def log_message(thread_id: str, role: str, content: str) -> None:
    """Persist one chat turn to the OAMP thread without blocking.

    The submit returns immediately; the actual `add_messages` call runs on
    a background thread. We drain completed futures opportunistically so
    failures surface on the next call rather than vanishing.
    """
    from oracleagentmemory.apis.thread import Message

    def _do_log():
        get_thread(thread_id).add_messages([Message(role=role, content=content)])

    _drain_log_errors()
    _LOG_PENDING.append(_LOG_EXECUTOR.submit(_do_log))


## Implement `agent_turn`

> 📖 **See:** [Part 5 guide → TODO 5](../docs/part-7-agent-loop.md#todo-5-implement-agent_turn)

The heart of the harness. Read the docs guide once before you start — it walks the dispatch pattern step-by-step. The complete solution is below; type it out, don't copy-paste, so you understand each line.


In [ ]:
# TODO 5: implement agent_turn(user_query, thread_id, max_iterations, budget_seconds, verbose)
# See docs/part-5-agent-loop.md for the full walkthrough.

def agent_turn(user_query: str, thread_id: str = "default",
               max_iterations: int = 8, budget_seconds: float = 360.0,
               verbose: bool = True) -> str:
    started = time.time()
    log_message(thread_id, "user", user_query)

    context = build_context(thread_id, user_query)
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message

        if not msg.tool_calls:
            final = msg.content or ""
            if verbose: print(f"  step {step}: final answer")
            break

        # Echo the assistant's tool_calls back so the LLM can match each
        # tool result to the call that produced it.
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        # Dispatch each tool the model asked for.
        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            if name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": output})

    # Forced final-answer path: if budget exhausted, ask one more time without tools.
    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)

    # Episodic memory — store the (user, assistant) pair for cross-thread recall.
    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240],
                      "elapsed_seconds": round(time.time() - started, 2)},
        )
    except Exception as _e:
        if verbose: print(f"  ! episodic add_memory failed: {type(_e).__name__}: {_e}")

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final

## Run the three-turn end-to-end demo

> 📖 **See:** [Part 5 guide → Demo: three-turn conversation](../docs/part-7-agent-loop.md#todo-8-run-the-three-turn-demo)

Three turns on the same thread:

1. **Discovery** — forces `search_knowledge` over scanned facts.
2. **Live data** — forces `run_sql`.
3. **Correction + persistence** — forces `remember` and creates a persisted correction memory.


In [ ]:
# Demo: three-turn conversation: run a three-turn conversation on a single thread.

thread = "demo-session-1"

q1 = "What's in the SUPPLYCHAIN schema? Briefly — list the entities and how they relate to each other."
print("USER:", q1)
print("\nASSISTANT:", agent_turn(q1, thread_id=thread))

In [ ]:
q2 = "How many active voyages does each carrier currently have? Show me a small table sorted by count desc."
print("USER:", q2)
print("\nASSISTANT:", agent_turn(q2, thread_id=thread))

In [ ]:
q3 = ("Important: in the SUPPLYCHAIN schema, vessels.capacity_teu is always in 20-foot equivalent units (TEU) — "
      "never tons, never cubic meters. And cargo_items.unit_value_cents is always USD CENTS, never dollars. "
      "Save EACH of these as a separate 'correction' memory by calling the remember tool BEFORE you respond, "
      "then confirm with the memory IDs you got back.")
print("USER:", q3)
print("\nASSISTANT:", agent_turn(q3, thread_id=thread))

# Part 6 — Identity-Aware Authorization with DDS

> 📖 **Guide:** [`docs/part-6-dds-identity.md`](../docs/part-8-dds-identity.md)

> 🔧 **TODO in this part:** **TODO 6** — `set_identity`

By default the agent sees every row and column the `AGENT` DB user has been granted. Real deployments need the trust boundary to follow the **end-user on whose behalf the agent is acting**, not the DB user the agent runs as.

**Oracle Deep Data Security (DDS)** moves that boundary into the database kernel. You declare row, column, and cell-level rules in declarative SQL, and the engine enforces them transparently — no matter what SQL the agent constructs.

```
end_user (real human)
       │  identity propagated via DBMS_SESSION.SET_CONTEXT
       ▼
AGENT (database user) ─── DDS policies ───▶ rows/cols filtered before tool sees them
```

Two demo identities:

| End-user | Authorized oceans | Clearance | Should see |
|---|---|---|---|
| `apac.fleet@acme.com` | PACIFIC + INDIAN | STANDARD | only those oceans; `unit_value_cents` masked to `NULL` |
| `cfo@acme.com` | ALL | EXECUTIVE | every voyage; declared cargo values visible |

> **What this workshop actually runs.** Real DDS in Oracle AI Database 26ai uses declarative DDL (`CREATE DATA SECURITY POLICY … USING (…)` / `… HIDE COLUMNS (…) WHEN (…)`), Data Grants for who-gets-what, and OAuth2/JWT-bound identity propagation. The Oracle AI Database Free image this Codespace ships with doesn't accept that DDL yet — so `setup_advanced.py` falls back to **`DBMS_RLS`** (the VPD-era policy engine that DDS supersedes). The trust boundary, the `EDA_CTX` propagation, and the observable result ("same SQL, two identities, different rows") are identical; only the policy DDL form differs. On a full DDS-capable 26ai image, the same `setup_advanced.py` lands real DDS policies. See [`docs/part-8-dds-identity.md`](../docs/part-8-dds-identity.md) for the side-by-side.


### Pre-built — DDS authorization tables, EDA_CTX namespace, policies

In [ ]:
# Tables that DDS policies will join against. End-users authorised by
# ocean region (PACIFIC / ATLANTIC / INDIAN / MEDITERRANEAN), with optional
# 'ALL' for executives / fleet-wide roles.
DDS_DDL = [
    ("agent_authorizations", (
        "CREATE TABLE agent_authorizations ("
        "  end_user     VARCHAR2(128) NOT NULL,"
        "  auth_region  VARCHAR2(20)  NOT NULL,"
        "  PRIMARY KEY (end_user, auth_region)"
        ")"
    )),
    ("agent_clearances", (
        "CREATE TABLE agent_clearances ("
        "  end_user   VARCHAR2(128) PRIMARY KEY,"
        "  clearance  VARCHAR2(32)  NOT NULL,"
        "  notes      VARCHAR2(400)"
        ")"
    )),
]

with agent_conn.cursor() as cur:
    for tname, ddl in DDS_DDL:
        try:
            cur.execute(ddl)
            print(f"created {tname}")
        except oracledb.DatabaseError as e:
            if e.args[0].code == 955:
                pass
            else:
                raise

    cur.execute("DELETE FROM agent_authorizations")
    cur.executemany(
        "INSERT INTO agent_authorizations (end_user, auth_region) VALUES (:1, :2)",
        [
            # Regional fleet managers
            ("apac.fleet@acme.com",      "PACIFIC"),
            ("apac.fleet@acme.com",      "INDIAN"),
            ("emea.fleet@acme.com",      "ATLANTIC"),
            ("emea.fleet@acme.com",      "MEDITERRANEAN"),
            ("americas.fleet@acme.com",  "PACIFIC"),
            ("americas.fleet@acme.com",  "ATLANTIC"),
            # C-suite — all oceans
            ("ceo@acme.com",             "ALL"),
            ("cfo@acme.com",             "ALL"),
        ],
    )
    cur.execute("DELETE FROM agent_clearances")
    cur.executemany(
        "INSERT INTO agent_clearances (end_user, clearance, notes) VALUES (:1, :2, :3)",
        [
            ("apac.fleet@acme.com",     "STANDARD",  "APAC + Indian Ocean fleet manager"),
            ("emea.fleet@acme.com",     "STANDARD",  "EMEA + Mediterranean fleet manager"),
            ("americas.fleet@acme.com", "STANDARD",  "Americas fleet manager"),
            ("ceo@acme.com",            "EXECUTIVE", "Chief Executive — sees declared values"),
            ("cfo@acme.com",            "EXECUTIVE", "Chief Financial Officer — sees declared values"),
        ],
    )
agent_conn.commit()

with sys_conn.cursor() as cur:
    cur.execute(f"GRANT SELECT ON {AGENT_USER}.agent_authorizations TO {DEMO_USER}")
    cur.execute(f"GRANT SELECT ON {AGENT_USER}.agent_clearances     TO {DEMO_USER}")
sys_conn.commit()
print("seeded authorizations + clearances; granted SELECT to SUPPLYCHAIN")


In [ ]:
# 1. Procedure that owns writes to EDA_CTX.
SETTER_SQL = """
CREATE OR REPLACE PROCEDURE set_eda_ctx(
    p_end_user  IN VARCHAR2,
    p_clearance IN VARCHAR2 DEFAULT NULL
) AS
    v_clearance VARCHAR2(32);
BEGIN
    -- Resolve clearance from the table if not explicitly passed.
    IF p_clearance IS NULL AND p_end_user IS NOT NULL THEN
        BEGIN
            SELECT clearance INTO v_clearance
              FROM agent_clearances WHERE end_user = p_end_user;
        EXCEPTION WHEN NO_DATA_FOUND THEN
            v_clearance := 'STANDARD';
        END;
    ELSE
        v_clearance := NVL(p_clearance, 'STANDARD');
    END IF;

    DBMS_SESSION.SET_CONTEXT('EDA_CTX', 'END_USER',  p_end_user);
    DBMS_SESSION.SET_CONTEXT('EDA_CTX', 'CLEARANCE', v_clearance);
END;
"""

with agent_conn.cursor() as cur:
    cur.execute(SETTER_SQL)
print("created procedure AGENT.set_eda_ctx")

# 2. Namespace bound to the setter. AGENT must have CREATE ANY CONTEXT
#    (granted via DB_DEVELOPER_ROLE in §3.2 - if not, do it as SYSDBA).
try:
    with agent_conn.cursor() as cur:
        cur.execute(f"CREATE OR REPLACE CONTEXT eda_ctx USING {AGENT_USER}.set_eda_ctx")
    print(f"created context EDA_CTX (writes restricted to {AGENT_USER}.set_eda_ctx)")
except oracledb.DatabaseError:
    with sys_conn.cursor() as cur:
        cur.execute(f"GRANT CREATE ANY CONTEXT TO {AGENT_USER}")
        cur.execute(f"CREATE OR REPLACE CONTEXT eda_ctx USING {AGENT_USER}.set_eda_ctx")
    print("granted CREATE ANY CONTEXT to AGENT and created EDA_CTX")

# 3. Also need DBMS_SESSION execute (already covered by RESOURCE/CONNECT).
# Smoke test: set + read back.
def set_identity(end_user: str | None, clearance: str | None = None) -> None:
    """Set the EDA_CTX identity on agent_conn for subsequent SQL."""
    with agent_conn.cursor() as cur:
        cur.callproc(f"{AGENT_USER}.set_eda_ctx", [end_user, clearance])

set_identity("cfo@acme.com")
with agent_conn.cursor() as cur:
    cur.execute("SELECT SYS_CONTEXT('EDA_CTX','END_USER'), SYS_CONTEXT('EDA_CTX','CLEARANCE') FROM DUAL")
    print("context after set:", cur.fetchone())

# Clear for the rest of setup; legacy paths see NULL = bypass.
set_identity(None)


In [ ]:
DDS_AVAILABLE = False
DDS_MODE = None

# ---------- Path A: declarative DDS (preferred, 26ai) ----------
ROW_POLICY_DDS = """
CREATE DATA SECURITY POLICY voyages_region_policy
ON voyages
USING (
    SYS_CONTEXT('EDA_CTX','END_USER') IS NULL
 OR EXISTS (
        SELECT 1 FROM AGENT.agent_authorizations a
         WHERE a.end_user = SYS_CONTEXT('EDA_CTX','END_USER')
           AND (a.auth_region = 'ALL' OR a.auth_region = voyages.ocean_region)
    )
)
"""

COL_POLICY_DDS = """
CREATE DATA SECURITY POLICY cargo_value_policy
ON cargo_items
HIDE COLUMNS (unit_value_cents)
WHEN SYS_CONTEXT('EDA_CTX','END_USER') IS NOT NULL
 AND COALESCE(SYS_CONTEXT('EDA_CTX','CLEARANCE'),'STANDARD') <> 'EXECUTIVE'
"""

for stmt in [
    "DROP DATA SECURITY POLICY voyages_region_policy",
    "DROP DATA SECURITY POLICY cargo_value_policy",
]:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(stmt)
    except oracledb.DatabaseError:
        pass

_FALLBACK_CODES = {900, 901, 922, 2000, 942, 1031}
dds_unsupported = False
created_count = 0
for label, ddl in [("row policy on voyages",       ROW_POLICY_DDS),
                   ("col policy on cargo_items",   COL_POLICY_DDS)]:
    try:
        with demo_conn.cursor() as cur:
            cur.execute(ddl)
        print(f"  declarative DDS: created {label}")
        created_count += 1
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        if code_ in _FALLBACK_CODES:
            print(f"  declarative DDS not available (ORA-{code_:05d}); will fall back to DBMS_RLS.")
            dds_unsupported = True
            break
        raise

if created_count == 2 and not dds_unsupported:
    DDS_AVAILABLE = True
    DDS_MODE = "declarative"

# ---------- Path B: DBMS_RLS fallback ----------
if dds_unsupported:
    with sys_conn.cursor() as cur:
        try:
            cur.execute(f"GRANT EXECUTE ON SYS.DBMS_RLS TO {DEMO_USER}")
            print(f"  granted EXECUTE ON DBMS_RLS to {DEMO_USER}")
        except oracledb.DatabaseError as e:
            if e.args[0].code != 1031:
                raise
    sys_conn.commit()

    DROP_BLOCK = """
DECLARE
  e_no_policy EXCEPTION;
  PRAGMA EXCEPTION_INIT(e_no_policy, -28102);
BEGIN
  BEGIN DBMS_RLS.DROP_POLICY(:s, 'VOYAGES',     'VOYAGES_REGION_POLICY');
  EXCEPTION WHEN e_no_policy THEN NULL; END;
  BEGIN DBMS_RLS.DROP_POLICY(:s, 'CARGO_ITEMS', 'CARGO_VALUE_POLICY');
  EXCEPTION WHEN e_no_policy THEN NULL; END;
END;
"""
    with demo_conn.cursor() as cur:
        try:
            cur.execute(DROP_BLOCK, s=DEMO_USER)
        except oracledb.DatabaseError:
            pass

    PRED_FN_VOYAGES = """
CREATE OR REPLACE FUNCTION voyages_region_predicate(
    p_schema IN VARCHAR2, p_object IN VARCHAR2
) RETURN VARCHAR2 AS
BEGIN
    -- Legacy / no identity = full visibility (so §13 demos still work).
    IF SYS_CONTEXT('EDA_CTX','END_USER') IS NULL THEN
        RETURN '1=1';
    END IF;
    -- Filter by the end-user's authorized ocean regions.
    RETURN q'[EXISTS (
        SELECT 1 FROM AGENT.agent_authorizations a
         WHERE a.end_user = SYS_CONTEXT('EDA_CTX','END_USER')
           AND (a.auth_region = 'ALL' OR a.auth_region = ocean_region)
    )]';
END;
"""

    PRED_FN_CARGO = """
CREATE OR REPLACE FUNCTION cargo_value_predicate(
    p_schema IN VARCHAR2, p_object IN VARCHAR2
) RETURN VARCHAR2 AS
BEGIN
    -- With sec_relevant_cols_opt => DBMS_RLS.ALL_ROWS, predicate-FALSE means
    -- "return the row but mask the listed columns (unit_value_cents) to NULL".
    --   * legacy / no identity     -> '1=1'  (no masking)
    --   * EXECUTIVE                 -> '1=1'  (no masking)
    --   * anyone else               -> '1=0'  (commercial value masked)
    IF SYS_CONTEXT('EDA_CTX','END_USER') IS NULL THEN
        RETURN '1=1';
    END IF;
    IF NVL(SYS_CONTEXT('EDA_CTX','CLEARANCE'),'STANDARD') = 'EXECUTIVE' THEN
        RETURN '1=1';
    END IF;
    RETURN '1=0';
END;
"""

    with demo_conn.cursor() as cur:
        cur.execute(PRED_FN_VOYAGES)
        cur.execute(PRED_FN_CARGO)
    print("  created predicate functions: voyages_region_predicate, cargo_value_predicate")

    ADD_ROW = """
BEGIN
  DBMS_RLS.ADD_POLICY(
    object_schema   => 'SUPPLYCHAIN',
    object_name     => 'VOYAGES',
    policy_name     => 'VOYAGES_REGION_POLICY',
    function_schema => 'SUPPLYCHAIN',
    policy_function => 'VOYAGES_REGION_PREDICATE',
    statement_types => 'SELECT'
  );
END;
"""
    ADD_COL = """
BEGIN
  DBMS_RLS.ADD_POLICY(
    object_schema         => 'SUPPLYCHAIN',
    object_name           => 'CARGO_ITEMS',
    policy_name           => 'CARGO_VALUE_POLICY',
    function_schema       => 'SUPPLYCHAIN',
    policy_function       => 'CARGO_VALUE_PREDICATE',
    statement_types       => 'SELECT',
    sec_relevant_cols     => 'UNIT_VALUE_CENTS',
    sec_relevant_cols_opt => DBMS_RLS.ALL_ROWS
  );
END;
"""
    with demo_conn.cursor() as cur:
        cur.execute(ADD_ROW)
        print("  added VPD policy VOYAGES_REGION_POLICY  (row-level on voyages.ocean_region)")
        cur.execute(ADD_COL)
        print("  added VPD policy CARGO_VALUE_POLICY     (column-mask on cargo_items.unit_value_cents)")

    DDS_AVAILABLE = True
    DDS_MODE = "dbms_rls"

print(f"\nDDS_AVAILABLE = {DDS_AVAILABLE}  (mode: {DDS_MODE})")


In [ ]:
import functools

# Wrap §10's tool_run_sql so an empty result set under a non-NULL identity
# is flagged. The model then knows to mention the identity restriction in its
# answer rather than claiming "there are no orders".
_inner_run_sql = TOOLS["run_sql"][0]

@register
def tool_run_sql(sql: str, max_rows: int = 50) -> str:
    """Execute a READ-ONLY SQL statement against the target Oracle AI Database.
    Results are filtered by Deep Data Security policies for the current end-user.
    If a SELECT returns 0 rows or NULL columns when an end-user identity is set,
    the result is annotated so the agent can explain the restriction.
    """
    raw = _inner_run_sql(sql, max_rows=max_rows)
    try:
        payload = json.loads(raw)
    except Exception:
        return raw
    if "error" in payload:
        return raw

    with agent_conn.cursor() as cur:
        cur.execute("SELECT SYS_CONTEXT('EDA_CTX','END_USER'), SYS_CONTEXT('EDA_CTX','CLEARANCE') FROM DUAL")
        end_user, clearance = cur.fetchone()

    if end_user:
        payload["dds_identity"] = {"end_user": end_user, "clearance": clearance}
        if payload.get("row_count", 0) == 0:
            payload["dds_note"] = (
                f"0 rows returned. Deep Data Security may be filtering on identity "
                f"end_user={end_user!r} clearance={clearance!r}."
            )
        # Heuristic: flag columns where every value is NULL — likely DDS column-hide.
        cols, rows = payload.get("columns", []), payload.get("rows", [])
        if cols and rows:
            null_cols = [
                cols[i] for i in range(len(cols))
                if all((row[i] is None) for row in rows)
            ]
            if null_cols:
                payload["dds_masked_columns"] = null_cols
    return json.dumps(payload, default=str)


# Redefine agent_turn to set EDA_CTX before the loop runs. Prior signature
# (no end_user) keeps working — when end_user is None, the setter clears the
# context and the legacy bypass in the row policy returns all rows.
_prior_agent_turn = agent_turn  # keep a reference in case anything else captures it

def agent_turn(user_query: str, thread_id: str = "default",
               max_iterations: int = 8, budget_seconds: float = 360.0,
               verbose: bool = True,
               end_user: str | None = None,
               clearance: str | None = None) -> str:
    """As before, plus per-turn identity propagation into Oracle DDS via EDA_CTX."""
    set_identity(end_user, clearance)
    if verbose and end_user:
        print(f"  [identity: end_user={end_user!r} clearance={clearance!r}]")
    try:
        return _prior_agent_turn(user_query, thread_id=thread_id,
                                 max_iterations=max_iterations,
                                 budget_seconds=budget_seconds,
                                 verbose=verbose)
    finally:
        set_identity(None)  # don't leak identity to the next caller

print("patched: tool_run_sql (DDS-aware result annotation), agent_turn (end_user/clearance params)")


### Same SQL, two identities — kernel-level filtering

In [ ]:
def probe(label: str, end_user: str | None, clearance: str | None, sql: str):
    print(f"--- {label} (end_user={end_user!r}, clearance={clearance!r}) ---")
    set_identity(end_user, clearance)
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql)
            cols = [d[0] for d in cur.description]
            rows = list(cur)
        print("  columns:", cols)
        for r in rows[:8]:
            print("   ", r)
        if len(rows) > 8:
            print(f"    ...({len(rows) - 8} more)")
        print(f"  row_count = {len(rows)}\n")
    finally:
        set_identity(None)


# Row-level filtering on voyages.ocean_region
probe("CEO sees all oceans", "ceo@acme.com", "EXECUTIVE",
      "SELECT ocean_region, COUNT(*) AS n FROM SUPPLYCHAIN.voyages GROUP BY ocean_region ORDER BY ocean_region")

probe("APAC fleet sees PACIFIC + INDIAN only", "apac.fleet@acme.com", "STANDARD",
      "SELECT ocean_region, COUNT(*) AS n FROM SUPPLYCHAIN.voyages GROUP BY ocean_region ORDER BY ocean_region")

probe("EMEA fleet sees ATLANTIC + MEDITERRANEAN only", "emea.fleet@acme.com", "STANDARD",
      "SELECT ocean_region, COUNT(*) AS n FROM SUPPLYCHAIN.voyages GROUP BY ocean_region ORDER BY ocean_region")

# Column-level masking on cargo_items.unit_value_cents
probe("CFO sees declared cargo values", "cfo@acme.com", "EXECUTIVE",
      "SELECT cargo_item_id, description, quantity, unit_value_cents FROM SUPPLYCHAIN.cargo_items FETCH FIRST 5 ROWS ONLY")

probe("APAC fleet: cargo values masked to NULL", "apac.fleet@acme.com", "STANDARD",
      "SELECT cargo_item_id, description, quantity, unit_value_cents FROM SUPPLYCHAIN.cargo_items FETCH FIRST 5 ROWS ONLY")

probe("Legacy / no identity (bypass)", None, None,
      "SELECT cargo_item_id, description, unit_value_cents FROM SUPPLYCHAIN.cargo_items FETCH FIRST 3 ROWS ONLY")


## Same SQL, two identities (direct probe)

Before the agent loop, the rock-solid version: same `SELECT`, same connection, only `set_identity(...)` changes between calls. If DDS is doing its job, the row policy and column mask rewrite the result set before it ever leaves the database — zero LLM involvement, zero memory pollution risk.


In [ ]:
SAME_SQL = '''
  SELECT v.ocean_region,
         COUNT(DISTINCT v.voyage_id) AS delayed_voyages,
         SUM(ci.unit_value_cents)    AS total_cargo_cents
    FROM SUPPLYCHAIN.voyages v
    JOIN SUPPLYCHAIN.containers c   ON c.voyage_id     = v.voyage_id
    JOIN SUPPLYCHAIN.cargo_items ci ON ci.container_id = c.container_id
   WHERE v.status = 'DELAYED'
   GROUP BY v.ocean_region
   ORDER BY v.ocean_region
'''

for persona, label in [
    (("cfo@acme.com",        "EXECUTIVE"), "CFO (EXECUTIVE, regions=ALL)"),
    (("apac.fleet@acme.com", "STANDARD"),  "apac.fleet (STANDARD, regions=PACIFIC+INDIAN)"),
]:
    set_identity(*persona)
    print("=" * 70)
    print(f"AS {label}")
    print("=" * 70)
    with agent_conn.cursor() as cur:
        cur.execute(SAME_SQL)
        cols = [d[0] for d in cur.description]
        print(f"  {cols[0]:14s} | {cols[1]:>15s} | {cols[2]:>20s}")
        print(f"  {'-'*14:14s} | {'-'*15:>15s} | {'-'*20:>20s}")
        for region, n, total in cur:
            total_str = f"{total:,}" if total is not None else "NULL (masked)"
            print(f"  {region:14s} | {n:>15d} | {total_str:>20s}")
    print()
set_identity(None)


## Run the same question with two identities

> 📖 **See:** [Part 6 guide → Demo: two-identity comparison](../docs/part-8-dds-identity.md#todo-9-run-the-same-question-with-two-identities)

Same natural-language question, asked twice with different `end_user` arguments. The agent constructs whatever SQL it likes; the database filters it.


In [ ]:
dds_thread_cfo  = f"demo-dds-cfo-{uuid.uuid4().hex[:6]}"
dds_thread_apac = f"demo-dds-apac-{uuid.uuid4().hex[:6]}"

q = ("For each ocean region, how many DELAYED voyages do we have, and what's "
     "the total of unit_value_cents across all their cargo items? Group by ocean_region.")

print("=" * 70)
print("ASKED AS CFO (clearance=EXECUTIVE, regions=ALL)")
print("=" * 70)
print(agent_turn(q, thread_id=dds_thread_cfo,
                 end_user="cfo@acme.com", clearance="EXECUTIVE"))

print("\n" + "=" * 70)
print("ASKED AS apac.fleet (clearance=STANDARD, regions=PACIFIC+INDIAN)")
print("=" * 70)
print(agent_turn(q, thread_id=dds_thread_apac,
                 end_user="apac.fleet@acme.com", clearance="STANDARD"))


# Part 7 — JSON Relational Duality Views

> 📖 **Guide:** [`docs/part-7-duality-views.md`](../docs/part-9-duality-views.md)

> 🔧 **TODO in this part:** **TODO 7** — register `tool_get_document`

A **duality view** is a JSON projection over a set of tables joined by PK/FK/UK relationships. The same row in `voyages` is accessible as a **relational tuple** *and* as a **nested JSON document** that includes its `vessel`, `carrier`, `origin`/`destination` ports, and the array of `containers` (with their `cargo_items` nested inside). One read, no JOINs, no client-side reshaping.

GPT-class models reason about JSON markedly better than tabular join results. And the Part 6 row policy on `voyages.ocean_region` is enforced *inside* the duality view by the kernel — same trust boundary, JSON shape on top.

![JSON Relational Duality — three lenses, one source of truth](../images/cover-duality-view.png)

### Pre-built — `voyage_dv` and `vessel_dv` DDL

Two read-only duality views on top of `SUPPLYCHAIN`. `voyage_dv` is the headline document; `vessel_dv` is fleet-centric (vessel + carrier + current position).

In [ ]:
DV_DDL = [
    "DROP VIEW IF EXISTS voyage_dv",
    "DROP VIEW IF EXISTS vessel_dv",

    # voyage_dv — the headline document. Read-only (no WITH UPDATE clause).
    """
    CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW voyage_dv AS
    SELECT JSON {
      '_id'         : v.voyage_id,
      'status'      : v.status,
      'oceanRegion' : v.ocean_region,
      'departureTs' : v.departure_ts,
      'etaTs'       : v.eta_ts,
      'origin'      : (
        SELECT JSON {
          'portCode' : po.port_code,
          'name'     : po.name,
          'country'  : po.country,
          'latitude' : po.latitude,
          'longitude': po.longitude
        } FROM ports po WHERE po.port_code = v.origin_code
      ),
      'destination' : (
        SELECT JSON {
          'portCode' : pd.port_code,
          'name'     : pd.name,
          'country'  : pd.country,
          'latitude' : pd.latitude,
          'longitude': pd.longitude
        } FROM ports pd WHERE pd.port_code = v.dest_code
      ),
      'vessel'      : (
        SELECT JSON {
          'vesselId'    : ve.vessel_id,
          'name'        : ve.name,
          'imoNumber'   : ve.imo_number,
          'vesselType'  : ve.vessel_type,
          'capacityTeu' : ve.capacity_teu,
          'flagCountry' : ve.flag_country,
          'carrier'     : (
            SELECT JSON {
              'carrierId' : ca.carrier_id,
              'name'      : ca.name,
              'hqCountry' : ca.hq_country
            } FROM carriers ca WHERE ca.carrier_id = ve.carrier_id
          )
        } FROM vessels ve WHERE ve.vessel_id = v.vessel_id
      ),
      'containers'  : [
        SELECT JSON {
          'containerId'   : c.container_id,
          'containerNo'   : c.container_no,
          'containerType' : c.container_type,
          'consignor'     : c.consignor,
          'consignee'     : c.consignee,
          'status'        : c.status,
          'cargo'         : [
            SELECT JSON {
              'cargoItemId'    : ci.cargo_item_id,
              'hsCode'         : ci.hs_code,
              'description'    : ci.description,
              'quantity'       : ci.quantity,
              'unitValueCents' : ci.unit_value_cents,
              'weightKg'       : ci.weight_kg
            } FROM cargo_items ci WHERE ci.container_id = c.container_id
          ]
        } FROM containers c WHERE c.voyage_id = v.voyage_id
      ]
    } FROM voyages v
    """,

    # vessel_dv — fleet-centric. Includes current position + active voyage if any.
    """
    CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW vessel_dv AS
    SELECT JSON {
      '_id'          : v.vessel_id,
      'name'         : v.name,
      'imoNumber'    : v.imo_number,
      'vesselType'   : v.vessel_type,
      'capacityTeu'  : v.capacity_teu,
      'yearBuilt'    : v.year_built,
      'flagCountry'  : v.flag_country,
      'carrier'      : (
        SELECT JSON {
          'carrierId' : c.carrier_id,
          'name'      : c.name,
          'hqCountry' : c.hq_country
        } FROM carriers c WHERE c.carrier_id = v.carrier_id
      ),
      'position'     : (
        SELECT JSON {
          'positionTs' : p.position_ts,
          'latitude'   : p.latitude,
          'longitude'  : p.longitude,
          'speedKnots' : p.speed_knots,
          'headingDeg' : p.heading_deg
        } FROM vessel_positions p WHERE p.vessel_id = v.vessel_id
      )
    } FROM vessels v
    """,
]

# DROP VIEW IF EXISTS isn't valid Oracle syntax — translate manually.
for stmt in DV_DDL:
    if stmt.strip().upper().startswith("DROP VIEW IF EXISTS"):
        view_name = stmt.split()[-1]
        try:
            with demo_conn.cursor() as cur:
                cur.execute(f"DROP VIEW {view_name}")
        except oracledb.DatabaseError:
            pass
        continue
    try:
        with demo_conn.cursor() as cur:
            cur.execute(stmt)
        # Identify which view we just created from the DDL
        head = stmt.strip().split("\n", 1)[0]
        print(f"OK: {head[:80]}")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        head = stmt.strip().split("\n", 1)[0][:80]
        if code_ in (900, 901, 922, 2000):
            print(f"!! {head!r}: duality-view syntax not on this image (ORA-{code_:05d}).")
            print("   The tool_get_document fallback below will return an error pointing")
            print("   the agent at run_sql instead.")
        else:
            raise
demo_conn.commit()

# Verify the views exist
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT view_name FROM all_views "
        " WHERE owner = :o AND view_name IN ('VOYAGE_DV', 'VESSEL_DV')",
        o=DEMO_USER,
    )
    print("duality views in", DEMO_USER, ":", [r[0] for r in cur])


## Register `tool_get_document`

> 📖 **See:** [Part 7 guide → TODO 7](../docs/part-9-duality-views.md#todo-7-register-tool_get_document)

Read one full document from a duality view by primary key. The agent calls this instead of writing JOINs whenever it needs the full shape of an entity.

`view` must be one of `voyage_dv` / `vessel_dv`. `key` is the value of `_id`. Return the JSON document as a string, or `{"error": ...}` if the view name is unknown or no document matches.


In [ ]:
# TODO 7: register tool_get_document with @register.
# Whitelist views to {"voyage_dv", "vessel_dv"}. Bind key as int when isdigit().
# Return JSON_SERIALIZE(data PRETTY) from SUPPLYCHAIN.<view>.

@register
def tool_get_document(view: str, key: str) -> str:
    """Read one full document from a JSON Relational Duality View by primary key.
    Use this instead of writing JOINs whenever you need the full shape of an entity
    (a voyage with its vessel/carrier/ports/containers/cargo, or a vessel with its
    carrier/position). Returns a JSON document.

    `view` must be one of: voyage_dv, vessel_dv.
    `key` is the value of the document _id (numeric voyage_id or vessel_id, as a string).
    """
    allowed = {"voyage_dv", "vessel_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                f"SELECT JSON_SERIALIZE(data PRETTY) FROM {DEMO_USER}.{view} "
                f" WHERE JSON_VALUE(data, '$._id') = :k",
                k=int(key) if str(key).isdigit() else key,
            )
            row = cur.fetchone()
        if not row:
            return json.dumps({"error": f"no document with _id={key} in {view}"})
        body = row[0].read() if hasattr(row[0], "read") else str(row[0])
        return body
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})

In [ ]:
# ✅ Checkpoint: TODO 7
assert "get_document" in TOOLS, "❌ TODO 7 incomplete — tool_get_document not registered"
out = tool_get_document(view="voyage_dv", key="1")
import json as _j
assert "_id" in out or "error" in out, "❌ Unexpected response shape"
print("✅ TODO 7 passed — tool_get_document registered")
print(out[:400])

### Pre-built — `tool_query_documents`

Filter a duality view with a SQL predicate. Pre-built so you can move on.

In [ ]:
@register
def tool_query_documents(view: str, where: str = "1=1", max_rows: int = 10) -> str:
    """Filter a JSON Relational Duality View with a SQL predicate.
    Use when you want a list of documents matching some condition without writing
    JOINs by hand. The predicate references underlying-table columns of the view's
    root table (e.g. status, ocean_region for voyage_dv; vessel_type for vessel_dv).

    `view` must be one of: voyage_dv, vessel_dv.
    `where` is a SQL boolean expression on the root table's columns (default '1=1').
    `max_rows` caps the result set.
    """
    allowed = {"voyage_dv", "vessel_dv"}
    if view not in allowed:
        return json.dumps({"error": f"unknown view {view!r}; allowed: {sorted(allowed)}"})
    sql = f"SELECT JSON_SERIALIZE(data) FROM {DEMO_USER}.{view} WHERE {where} FETCH FIRST :n ROWS ONLY"
    try:
        with agent_conn.cursor() as cur:
            cur.execute(sql, n=max_rows)
            docs = [(r[0].read() if hasattr(r[0], "read") else str(r[0])) for r in cur]
        return json.dumps({"count": len(docs), "documents": [json.loads(d) for d in docs]}, default=str)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}", "sql": sql})


print(f"registered: query_documents  (TOOLS total: {len(TOOLS)})")

### Demo — same question, the duality-view path

Ask a "give me everything about voyage 7" question. Watch the trace — the agent should reach for `get_document("voyage_dv", "7")` and get the full nested document back in one tool call.

In [ ]:
demo_thread_dv = "demo-dv-1"

q_dv = (
    "Give me a complete picture of voyage_id 7: which carrier and vessel is running it, "
    "what ports it's between, what's loaded on it (containers and cargo items), and the "
    "vessel's current position. Use the duality view if you have one — that's why we built it."
)

print("=" * 70)
print(q_dv)
print("=" * 70)
print(agent_turn(q_dv, thread_id=demo_thread_dv))


### Pre-built — writable view + ETag conflict demo

`voyage_status_dv` adds `WITH UPDATE` to the DV definition, making it writable. Every retrieved document carries `_metadata.etag`; stale writes raise `ORA-42699` automatically. The next two cells demonstrate a clean round-trip and a deliberate two-writer conflict.

In [ ]:
# A SECOND duality view, dedicated to the writable demo.
# Only voyage_id (the key) and status are exposed — keeping the surface narrow
# limits what an UPDATE through this view can change. ocean_region stays
# read-only so the §14.4 row policy can't be circumvented by mutating it
# through the JSON layer.
DV_WRITABLE_DDL = """
CREATE OR REPLACE JSON RELATIONAL DUALITY VIEW voyage_status_dv AS
SELECT JSON {
  '_id'         : v.voyage_id,
  'status'      : v.status,
  'oceanRegion' : v.ocean_region,
  'departureTs' : v.departure_ts,
  'etaTs'       : v.eta_ts
} FROM voyages v WITH UPDATE
"""

try:
    with demo_conn.cursor() as cur:
        cur.execute(DV_WRITABLE_DDL)
        # AGENT was granted SELECT ANY TABLE in §4 so reads work, but UPDATE
        # through a duality view requires the explicit UPDATE privilege on
        # the view itself (ORA-41900 otherwise). Owner side grants it here.
        cur.execute(f"GRANT SELECT, UPDATE ON voyage_status_dv TO {AGENT_USER}")
    demo_conn.commit()
    print(f"OK: created voyage_status_dv WITH UPDATE; granted UPDATE to {AGENT_USER}")
except oracledb.DatabaseError as e:
    code_ = e.args[0].code
    if code_ in (900, 901, 922, 2000):
        print(f"!! duality-view DDL not supported on this image (ORA-{code_:05d}). "
              "The §11.6.4 demos below will skip if voyage_status_dv isn't present.")
    else:
        raise

# Quick check that it landed
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT view_name FROM all_views WHERE owner = :o AND view_name = 'VOYAGE_STATUS_DV'",
        o=DEMO_USER)
    rows = list(cur)
print(f"voyage_status_dv exists: {bool(rows)}")


In [ ]:
import json as _json_dv

# Pick a voyage to round-trip. We capture the initial status so the cell is
# idempotent — we restore it at the end and the rest of the notebook sees no
# side effect.
TARGET_VOYAGE = 1


def _read_doc(voyage_id):
    """Fetch one document from voyage_status_dv. The result includes
    _metadata.etag, which the kernel sets per-row."""
    with agent_conn.cursor() as cur:
        cur.execute(
            # Aliasing the view as `t` is required so Oracle parses
            # `t.data."_id"` as a JSON path step rather than ambiguous
            # <table>.<column> syntax (which raises ORA-00904).
            f"SELECT JSON_SERIALIZE(t.data PRETTY) "
            f"  FROM {DEMO_USER}.voyage_status_dv t "
            f" WHERE JSON_VALUE(t.data, '$._id') = :k",
            k=voyage_id,
        )
        row = cur.fetchone()
    if not row:
        return None
    raw = row[0]
    if hasattr(raw, "read"):
        raw = raw.read()
    return _json_dv.loads(raw)


def _put_doc(voyage_id, doc):
    """PUT a modified document back through the DV. The kernel compares the
    doc's _metadata.etag against the row's current etag and rejects the UPDATE
    with ORA-42699 if they differ."""
    with agent_conn.cursor() as cur:
        cur.execute(
            f"UPDATE {DEMO_USER}.voyage_status_dv t "
            f"   SET t.data = :new_doc "
            f" WHERE JSON_VALUE(t.data, '$._id') = :k",
            new_doc=_json_dv.dumps(doc), k=voyage_id,
        )
        rc = cur.rowcount
    agent_conn.commit()
    return rc


# -- Read --
doc = _read_doc(TARGET_VOYAGE)
if doc is None:
    print(f"voyage {TARGET_VOYAGE} not found — nothing to demo")
else:
    initial_status = doc["status"]
    initial_etag = doc.get("_metadata", {}).get("etag")
    print(f"Initial state of voyage {TARGET_VOYAGE}:")
    print(f"  status: {initial_status}")
    print(f"  etag:   {initial_etag}")

    # -- Modify in memory --
    doc["status"] = "DELAYED" if initial_status != "DELAYED" else "ACTIVE"
    new_status = doc["status"]
    print(f"\nFlipping status -> {new_status} and PUTting back with the matching etag...")

    # -- PUT back --
    try:
        rc = _put_doc(TARGET_VOYAGE, doc)
        print(f"  rows updated: {rc}")
    except oracledb.DatabaseError as e:
        print(f"  PUT failed: {e}")
        rc = 0

    # -- Verify --
    after = _read_doc(TARGET_VOYAGE)
    if after:
        print(f"\nAfter PUT — status: {after['status']}, "
              f"new etag: {after.get('_metadata', {}).get('etag')}")
        print(f"  (etag changed: {after.get('_metadata', {}).get('etag') != initial_etag})")

    # -- Restore (so the rest of the notebook sees no drift) --
    after["status"] = initial_status
    _put_doc(TARGET_VOYAGE, after)
    print(f"\nRestored status to {initial_status} (cell is idempotent on re-run).")


In [ ]:
# Conflict demo: two readers grab the same doc. First writer commits, second
# writer's ETag is now stale and the kernel rejects the UPDATE.
TARGET_VOYAGE = 2

original = _read_doc(TARGET_VOYAGE)
if original is None:
    print(f"voyage {TARGET_VOYAGE} not found — nothing to demo")
else:
    initial_status = original["status"]
    print(f"Initial state of voyage {TARGET_VOYAGE}: status={initial_status}, "
          f"etag={original.get('_metadata', {}).get('etag')}")

    # Two readers, same etag (because they read at the same logical state).
    reader_a = _read_doc(TARGET_VOYAGE)
    reader_b = _read_doc(TARGET_VOYAGE)
    print(f"\nReader A and Reader B both have etag={reader_a.get('_metadata',{}).get('etag')}")

    # Writer A commits first — succeeds.
    reader_a["status"] = "DELAYED" if initial_status != "DELAYED" else "ACTIVE"
    print(f"\nWriter A: PUT status={reader_a['status']}")
    try:
        rc = _put_doc(TARGET_VOYAGE, reader_a)
        print(f"  rows updated: {rc}  (success — etag matched)")
    except oracledb.DatabaseError as e:
        print(f"  unexpected failure: {e}")

    # Writer B tries to PUT with the *original* (now stale) etag — should fail.
    reader_b["status"] = "COMPLETED"
    print(f"\nWriter B: PUT status={reader_b['status']} (with stale etag)")
    try:
        rc = _put_doc(TARGET_VOYAGE, reader_b)
        print(f"  rows updated: {rc}  (this should NOT have succeeded)")
    except oracledb.DatabaseError as e:
        code_ = e.args[0].code
        if code_ == 42699:
            print(f"  REJECTED with ORA-42699 — stale etag, exactly as expected.")
            print(f"  Writer B would now read again, re-apply its change, and retry.")
        else:
            print(f"  REJECTED with ORA-{code_:05d}: {e}")

    # Restore for cell idempotency
    final = _read_doc(TARGET_VOYAGE)
    final["status"] = initial_status
    _put_doc(TARGET_VOYAGE, final)
    print(f"\nRestored status of voyage {TARGET_VOYAGE} to {initial_status}.")


# Part 8 — Continuous Scans via DBMS_SCHEDULER

> 📖 **Guide:** [`docs/part-8-scheduler.md`](../docs/part-10-scheduler.md)

Schemas drift; institutional knowledge must drift with them. The scanner from Part 2 is cheap (the `body_hash` check makes re-scans free), but we still need *something* to trigger it on a cadence. The natural place for that schedule is **inside the database** via `DBMS_SCHEDULER`.

We use the scheduler as a **trigger**: a tiny PL/SQL proc inserts a `queued-by-scheduler` row into `scan_history`. The harness, when next running, calls `drain_queued_scans()` which picks up queued rows and runs `run_scan(owner)` on each. Python logic stays in one place; the database owns the schedule.

In [ ]:
# 8.1 — Opt in to scheduler setup. Set RUN_SCHEDULER=1 in the env to install
# the job; otherwise the cells below skip the install.

os.environ.setdefault("RUN_SCHEDULER", "1")

### Pre-built — migrate `scan_history.notes` from CLOB → VARCHAR2

`ORA-22848` forbids CLOB equality comparisons, but the queue check is `WHERE notes = 'queued-by-scheduler'`. Idempotent: only runs if the column is still CLOB.

In [ ]:
# Migrate existing scan_history.notes (CLOB) -> VARCHAR2(4000) so the
# `WHERE notes = 'queued-by-scheduler'` lookup below works (ORA-22848: cannot
# use CLOB type as comparison key).  Idempotent: only runs if the column is
# still CLOB.
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT data_type FROM user_tab_columns "
        " WHERE table_name = 'SCAN_HISTORY' AND column_name = 'NOTES'"
    )
    row = cur.fetchone()
    if row and row[0] == 'CLOB':
        cur.execute("ALTER TABLE scan_history RENAME COLUMN notes TO notes_clob")
        cur.execute("ALTER TABLE scan_history ADD notes VARCHAR2(4000)")
        cur.execute("UPDATE scan_history SET notes = DBMS_LOB.SUBSTR(notes_clob, 4000, 1)")
        cur.execute("ALTER TABLE scan_history DROP COLUMN notes_clob")
        agent_conn.commit()
        print("migrated scan_history.notes  CLOB -> VARCHAR2(4000)")
    else:
        print(f"scan_history.notes already {row[0] if row else '?'}; no migration needed")


### Pre-built — install the scheduler job + drain helper

Creates `AGENT_REQUEST_SCAN(p_owner)` (the proc the scheduler calls), schedules it via `DBMS_SCHEDULER.CREATE_JOB`, and defines `drain_queued_scans()` for the harness to consume the queue.

In [ ]:
import os

if os.environ.get("RUN_SCHEDULER", "0") != "1":
    print("Skipping DBMS_SCHEDULER setup. Set RUN_SCHEDULER=1 to install it.")
else:
    SCAN_PROC = "AGENT_REQUEST_SCAN"
    SCAN_JOB  = "AGENT_PERIODIC_SCAN"
    SCAN_INTERVAL_MIN = int(os.environ.get("SCAN_INTERVAL_MIN", "60"))
    SCAN_OWNER        = os.environ.get("SCAN_OWNER", DEMO_USER)

    # 1. Create the stored proc that records a "scan requested" row in scan_history.
    #    The harness side drains queued rows below; once consumed, run_scan rewrites the
    #    same row with the actual outcome.
    proc_sql = f"""
    CREATE OR REPLACE PROCEDURE {SCAN_PROC}(p_owner IN VARCHAR2) AS
    BEGIN
      INSERT INTO scan_history (target_owner, notes)
      VALUES (UPPER(p_owner), 'queued-by-scheduler');
      COMMIT;
    END;
    """
    with agent_conn.cursor() as cur:
        cur.execute(proc_sql)
    print(f"created procedure {SCAN_PROC}")

    # 2. (Re-)create the scheduler job.
    with agent_conn.cursor() as cur:
        # Drop existing if any (idempotent).
        try:
            cur.execute("BEGIN DBMS_SCHEDULER.DROP_JOB(:n, force => TRUE); END;",
                        n=SCAN_JOB)
        except oracledb.DatabaseError:
            pass

        cur.execute(
            "BEGIN "
            "  DBMS_SCHEDULER.CREATE_JOB("
            "    job_name        => :job_name, "
            "    job_type        => 'STORED_PROCEDURE', "
            "    job_action      => :proc, "
            "    number_of_arguments => 1, "
            "    start_date      => SYSTIMESTAMP, "
            "    repeat_interval => :rep, "
            "    enabled         => FALSE); "
            "  DBMS_SCHEDULER.SET_JOB_ARGUMENT_VALUE("
            "    job_name => :job_name, argument_position => 1, argument_value => :owner); "
            "  DBMS_SCHEDULER.ENABLE(:job_name); "
            "END;",
            job_name=SCAN_JOB,
            proc=SCAN_PROC,
            rep=f"FREQ=MINUTELY; INTERVAL={SCAN_INTERVAL_MIN}",
            owner=SCAN_OWNER,
        )
    agent_conn.commit()
    print(f"scheduled job {SCAN_JOB} -> {SCAN_PROC}({SCAN_OWNER!r}) every {SCAN_INTERVAL_MIN} min")


# Harness-side drain: run scans for any queued requests since last call.
# Call this from your agent loop, a worker process, or just inline before the demo.
def drain_queued_scans(verbose: bool = True) -> int:
    """Process every 'queued-by-scheduler' row in scan_history.

    For each queued row, run the Python scanner (`run_scan`) and update the row's
    notes/finished_at fields. Returns the number of scans actually run.
    """
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT scan_id, target_owner FROM scan_history "
            " WHERE notes = 'queued-by-scheduler' "
            "   AND finished_at IS NULL "
            " ORDER BY started_at"
        )
        queued = list(cur)

    ran = 0
    for scan_id, owner in queued:
        try:
            summary = run_scan(agent_conn, owner=owner)
            with agent_conn.cursor() as cur:
                cur.execute(
                    "UPDATE scan_history SET notes = :n WHERE scan_id = :id",
                    n=f"drained: new={summary['new']} updated={summary['updated']} "
                      f"skipped={summary['skipped']}",
                    id=scan_id,
                )
            agent_conn.commit()
            ran += 1
            if verbose: print(f"[drain] scanned {owner}: {summary}")
        except Exception as e:
            print(f"[drain] {owner} failed: {e}")
    return ran


# Sanity: how many queued requests are pending right now?
with agent_conn.cursor() as cur:
    cur.execute(
        "SELECT COUNT(*) FROM scan_history "
        " WHERE notes = 'queued-by-scheduler' AND finished_at IS NULL"
    )
    pending = cur.fetchone()[0]
print(f"queued scans pending: {pending} (run drain_queued_scans() to process)")


# Part 9 — Tool-Output Offload

> 📖 **Guide:** [`docs/part-9-tool-output-offload.md`](../docs/part-11-tool-output-offload.md)

> 🔧 **TODOs in this part (2):** **TODO 8** — `log_tool`; **TODO 9** — register `tool_fetch_tool_output`

Part 5's `agent_turn` inlines every tool result verbatim. Fine for short outputs, **but blows the context window** on a 50-row `run_sql` or a multi-KB skill body. Part 9 fixes that with three pieces:

1. `log_tool` — every dispatch persists the full output as an OAMP memory tagged `kind=tool_output` with the LLM's `tool_call_id`.
2. **Truncation marker** — outputs over 600 bytes are replaced in the message list with a compact preview ending in `...[+N bytes. full output: fetch_tool_output(tool_call_id='call_…')]`.
3. `fetch_tool_output(tool_call_id)` — your TODO 9 — the agent-side retrieval tool that recovers the full bytes by id.

After running this section, `agent_turn` is **redefined** to use the offload + truncation pattern. To revert to the minimal version from Part 5, re-run the Part 5 `agent_turn` cell.

### Pre-built — `log_tool`

Persist the full tool output as an OAMP memory tagged with the `tool_call_id`.

In [ ]:
def log_tool(thread_id: str, tool_call_id: str, tool_name: str,
             tool_args: dict, tool_output: str) -> None:
    """Offload one tool result to OAMP as a kind=tool_output memory."""
    memory_client.add_memory(
        tool_output,
        user_id=USER_ID, agent_id=AGENT_ID,
        thread_id=thread_id,
        metadata={
            "kind": "tool_output",
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "tool_args": json.dumps(tool_args),
        },
    )
print("log_tool ready")

## Register `tool_fetch_tool_output`

> 📖 **See:** [Part 9 guide → TODO 9](../docs/part-11-tool-output-offload.md#todo-9-register-tool_fetch_tool_output)

The agent-side retrieval tool. The agent calls it when its inlined preview was truncated and it needs the missing bytes to answer.

Use `memory_client._store.list(..., metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id}, limit=1)` and return a JSON object with `tool_name`, `tool_args`, `tool_output` if found, or `{"error": ...}` if no record matches.


In [ ]:
# TODO 9: register tool_fetch_tool_output with @register.

@register
def tool_fetch_tool_output(tool_call_id: str) -> str:
    """Retrieve the full, untruncated output of a previous tool call.
    Use this when a prior tool result in your context was truncated with
    '...[+N bytes. full output: fetch_tool_output(tool_call_id=...)]' and you need
    the missing bytes to answer.
    """
    records = memory_client._store.list(
        "memory",
        user_id=USER_ID, agent_id=AGENT_ID,
        metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id},
        limit=1,
    )
    if not records:
        return json.dumps({"error": f"no tool call with id {tool_call_id}"})
    r = records[0]
    meta = r.metadata or {}
    return json.dumps({
        "tool_name":   meta.get("tool_name"),
        "tool_args":   meta.get("tool_args"),
        "tool_output": r.content,
    })

In [ ]:
# ✅ Checkpoint: TODO 9
assert "fetch_tool_output" in TOOLS, "❌ TODO 9 incomplete — tool_fetch_tool_output not registered"
print("✅ TODO 9 passed — tool_fetch_tool_output registered")

### Pre-built — redefine `agent_turn` with offload + truncation marker

Same loop, but every dispatch goes through `log_tool` (full output → OAMP) and the message list gets a compact preview with a recovery hint when output > 600 bytes.

In [ ]:
# §14.2 — tool-output offload (the missing piece in §11's minimal agent_turn).
#
# Three changes wired together:
#   (a) `log_tool` — persist the full tool output as an OAMP memory tagged
#       kind=tool_output, indexed by the LLM's tool_call_id;
#   (b) `tool_fetch_tool_output` — the agent-side retrieval tool that pulls
#       the bytes back by id when its inlined preview was truncated;
#   (c) redefined `agent_turn` — calls log_tool for each dispatch and replaces
#       large outputs in the message list with a truncation marker that points
#       the agent at fetch_tool_output.
#
# Once you run this cell, all subsequent turns offload + truncate. Re-run
# cell §11's `agent_turn` to revert to the minimal verbatim-inline version.

def log_tool(thread_id: str, tool_call_id: str, tool_name: str,
             tool_args: dict, tool_output: str) -> None:
    """Offload one tool result to OAMP as a kind=tool_output memory."""
    memory_client.add_memory(
        tool_output,
        user_id=USER_ID, agent_id=AGENT_ID,
        thread_id=thread_id,
        metadata={
            "kind": "tool_output",
            "tool_call_id": tool_call_id,
            "tool_name": tool_name,
            "tool_args": json.dumps(tool_args),
        },
    )


@register
def tool_fetch_tool_output(tool_call_id: str) -> str:
    """Retrieve the full, untruncated output of a previous tool call.
    Use this when a prior tool result in your context was truncated with
    '...[+N bytes. full output: fetch_tool_output(tool_call_id=...)]' and you need
    the missing bytes to answer.
    """
    records = memory_client._store.list(
        "memory",
        user_id=USER_ID, agent_id=AGENT_ID,
        metadata_filter={"kind": "tool_output", "tool_call_id": tool_call_id},
        limit=1,
    )
    if not records:
        return json.dumps({"error": f"no tool call with id {tool_call_id}"})
    r = records[0]
    meta = r.metadata or {}
    return json.dumps({
        "tool_name":   meta.get("tool_name"),
        "tool_args":   meta.get("tool_args"),
        "tool_output": r.content,
    })


# Redefine agent_turn to add the offload + truncation marker.
def agent_turn(user_query: str, thread_id: str = "default",     # noqa: F811
                max_iterations: int = 8, budget_seconds: float = 360.0,
                verbose: bool = True) -> str:
    started = time.time()
    log_message(thread_id, "user", user_query)
    context = build_context(thread_id, user_query)
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": context},
    ]
    tool_schemas = retrieve_tools(user_query, k=6)

    final = ""
    step = 0
    DEDUPE_WINDOW = 3
    recent_calls: list[tuple[str, str]] = []
    for step in range(max_iterations):
        if time.time() - started > budget_seconds:
            if verbose: print(f"  ! budget exhausted at iteration {step}")
            break

        resp = chat(messages, tools=tool_schemas)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            final = msg.content or ""
            if verbose: print(f"  step {step}: final answer")
            break

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name,
                              "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            if verbose: print(f"  step {step}: -> {name}({args})")

            # Dedupe: short-circuit if the LLM is dispatching the same
            # (tool, args) it called within the last DEDUPE_WINDOW steps.
            # Common pathology: agent loops re-reading the same scratchpad
            # path or re-issuing the same search_knowledge query, never
            # converging until budget exhausts.
            call_key = (name, json.dumps(args, sort_keys=True))
            if call_key in recent_calls[-DEDUPE_WINDOW:]:
                output = json.dumps({
                    "note": f"duplicate call to {name} with identical args within "
                            f"the last {DEDUPE_WINDOW} dispatches — output unchanged; "
                            f"reuse the prior tool result above instead of re-dispatching."
                })
                if verbose: print(f"          ↳ duplicate dispatch — short-circuited")
            elif name not in TOOLS:
                output = json.dumps({"error": f"unknown tool: {name}"})
            else:
                fn, _ = TOOLS[name]
                try:
                    output = fn(**args)
                except Exception as e:
                    output = json.dumps({"error": f"{type(e).__name__}: {e}"})
            recent_calls.append(call_key)

            # Offload full bytes, inline a compact preview with a recovery hint.
            log_tool(thread_id, tc.id, name, args, output)
            if len(output) <= 600:
                preview = output
            else:
                preview = (
                    output[:600] +
                    f" ...[+{len(output)-600} bytes. "
                    f"full output: fetch_tool_output(tool_call_id='{tc.id}')]"
                )
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": preview})

    if not final:
        messages.append({"role": "user",
                         "content": "Budget exhausted. Provide your best answer now, no more tools."})
        resp = chat(messages, tools=None)
        final = resp.choices[0].message.content or "(no answer produced)"

    log_message(thread_id, "assistant", final)

    try:
        memory_client.add_memory(
            f"User: {user_query}\n\nAssistant: {final}",
            user_id=USER_ID, agent_id=AGENT_ID,
            thread_id=thread_id,
            metadata={"kind": "episodic", "thread_id": thread_id,
                      "user_query": user_query[:240],
                      "elapsed_seconds": round(time.time() - started, 2)},
        )
    except Exception as _e:
        if verbose: print(f"  ! episodic add_memory failed: {type(_e).__name__}: {_e}")

    if verbose: print(f"  [{time.time() - started:.1f}s, {step + 1} steps]")
    return final


# Sanity: ask for the full output of the most recent tool call on `thread`.
recent = memory_client._store.list(
    "memory",
    user_id=USER_ID, agent_id=AGENT_ID,
    thread_id=thread,
    metadata_filter={"kind": "tool_output"},
    limit=1,
)
if recent:
    last_id = (recent[0].metadata or {}).get("tool_call_id", "")
    print(f"fetching tool_call_id={last_id}")
    print(tool_fetch_tool_output(tool_call_id=last_id)[:400])
else:
    print("(no tool_output memories yet on this thread — run a turn first)")


# Closing thoughts — and the running app

You've built and exercised every component of the harness:

- Long-term memory survives between turns and across sessions, because OAMP persists in Oracle.
- Retrieval combines semantic similarity (vector) with exact-token precision (Oracle Text), fused via RRF in one SQL statement.
- Tools are vector-indexed; the registry can grow past 30 tools without bloating the per-turn prompt.
- The agent dispatches `run_sql`, `exec_js`, `scratch_*`, and `remember` through a 90-line loop.
- Identity-aware authorization is enforced by the database kernel — the agent's grants no longer determine what the end-user sees.
- JSON Relational Duality Views serve the same row as a tuple AND a nested document, with row policies still enforced.
- A `DBMS_SCHEDULER` job drives continuous re-scans so institutional knowledge tracks reality.
- Tool-output offload + truncation markers keep the context window stable on long, data-heavy turns.

## See it running — open the app

The Codespace started a Flask + React deployment of this same harness on first launch. Open it now:

> **http://localhost:3000** (auto-forwarded — check the **PORTS** tab at the bottom of VS Code if it didn't open in a preview)

The app uses the same Oracle, the same OAMP store, the same `toolbox` and `skillbox` you populated in this notebook. What's different:

- A real chat UI, with each tool dispatch shown as a separate bubble.
- A live **memory pane** on the right — top semantic memories, recent tool outputs, skill manifest, token usage — refreshed after every turn.
- An **identity selector** in the header — pick a persona (`cfo`, `analyst.east`, `analyst.west`, `ops.viewer`) and the same SQL returns different rows.
- A **3D globe** the agent can drive via the `focus_world` tool.
- Live web/news access via the `search_tavily` tool (set `TAVILY_API_KEY` to enable).

### Try these prompts

| Prompt | Exercises |
|---|---|
| *"What's in the SUPPLYCHAIN schema?"* | scanner-built institutional knowledge (Part 2) |
| *"How many active voyages does each carrier have?"* | `run_sql` (Part 4 + Part 5) |
| *"Pull the lat/long of every in-transit vessel and use exec_js to compute the haversine distance from the fleet centroid for each."* | `run_sql` → `exec_js` (Oracle MLE) |
| *"Give me the complete document for voyage 7 — use the duality view."* | `get_document("voyage_dv", "7")` (Part 7) |
| *"How do I diagnose ORA-00904? Consult any guide you have."* | `load_skill` from the skillbox |

For the full app architecture, read [`app/README.md`](../app/README.md).

## What's left if you want to go further

- **Replace the regex SQL guard with a real parser.** `antlr` with Oracle's grammar, or a stage that runs `EXPLAIN PLAN FOR` and mines `plan_table` for accessed objects.
- **Layer evaluations on top.** Pair each golden question with a target SQL, execute both, grade with the LLM. Regressions surface as failed grades in CI.
- **Tune OAMP's extraction cadence.** `memory_extraction_frequency` and `context_summary_update_frequency` trade fidelity against LLM cost.
- **Push policy further into the kernel.** The Part 6 DDS row policy + Part 7 duality views compose: try adding writable duality views with end-user-aware constraints (only `cfo` can mark voyages COMPLETED, etc).

The harness is small on purpose. Make it boring, make it legible, and let the model do the interesting work.